# Climate Baseline — Pre-2024 Fit

**Author:** Muhammad Kaleem — Climate Module Lead  
**Project:** PLEXA — Multi-Modal Smart City Digital Twin and Cascading Resilience Engine

---

This notebook prepares a new climate baseline for the 2024 historical
simulated-live prototype using chronological pre-2024 training and
development data.

### Objective

Predict whether a physical street segment receives a qualifying street
or highway flooding report during the next hour.

### Model Inputs

Retain the existing ten features: four terrain/length features, four
rainfall accumulations and two historical-report features.

### Experiment Plan

- Record training/development dates and label-availability rules before fitting.
- Keep 2024 outcomes out of training, tuning and the frozen history pool.
- Assess the 700 requested mobility roads, preserving mapping uncertainty.
- Check the exact 2024 hourly window and retain missing rainfall as unavailable.
- Freeze the new model and threshold before generating replay outputs.
- Preserve the existing model release and verified Ida results.

### Interpretation

Outputs are uncalibrated flooding-report scores, not water depth, road
closure or safe-passage predictions. Previous use of 2024 data prevents
claiming a pristine independent evaluation.

---

In [2]:
import hashlib
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    (
        path for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "config/climate_preparation.json").is_file()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the project root.")

REQUEST_DIR = (
    Path.home()
    / "Downloads/climate_2024_coverage_request_v0.1.0"
)

if not REQUEST_DIR.is_dir():
    raise FileNotFoundError(REQUEST_DIR)

print("Project root:", PROJECT_ROOT)
print("Request folder:", REQUEST_DIR)

Project root: /Users/macbookairm3/Documents/Plexa-A-Multi-Modal-Smart-City-Digital-Twin-and-Cascading-Resilience-Engine
Request folder: /Users/macbookairm3/Downloads/climate_2024_coverage_request_v0.1.0


## 1. Request Verification

Verify the supplied files, road identities and exact scenario window
before assessing mappings or preparing training data.

In [3]:
request_manifest = json.loads(
    (REQUEST_DIR / "manifest.json").read_text()
)

verification_rows = []

for name, expected in request_manifest["files"].items():
    # Restrict manifest entries to files directly inside the request folder.
    if Path(name).name != name:
        raise ValueError(f"Unexpected manifest path: {name}")

    path = REQUEST_DIR / name

    if not path.is_file():
        raise FileNotFoundError(path)

    content = path.read_bytes()
    actual_hash = hashlib.sha256(content).hexdigest()

    if (
        len(content) != expected["bytes"]
        or actual_hash != expected["sha256"]
    ):
        raise ValueError(f"Verification failed: {name}")

    verification_rows.append({
        "file": name,
        "status": "MATCH",
    })

display(pd.DataFrame(verification_rows))
print("Request files verified. No files changed.")

,file,status
0,requested_mobility_roads.geojson,MATCH
1,supported_mobility_contexts.csv,MATCH
2,geometry_exclusions.csv,MATCH
3,request_spec.json,MATCH
4,message_for_kaleem.txt,MATCH
5,README.md,MATCH


Request files verified. No files changed.


## 2. Requested Roads and Mobility Contexts

Load the supplied road identities and serving contexts.
Geometry matching will be assessed separately.

In [4]:
request_spec = json.loads(
    (REQUEST_DIR / "request_spec.json").read_text()
)

requested_geojson = json.loads(
    (REQUEST_DIR / "requested_mobility_roads.geojson").read_text()
)

requested_roads = pd.DataFrame([
    feature["properties"]
    for feature in requested_geojson["features"]
])

mobility_contexts = pd.read_csv(
    REQUEST_DIR / "supported_mobility_contexts.csv",
    dtype=str,
)

geometry_exclusions = pd.read_csv(
    REQUEST_DIR / "geometry_exclusions.csv",
    dtype=str,
)

display(pd.DataFrame({
    "Item": [
        "Road geometries supplied",
        "Mobility road/direction contexts",
        "Roads excluded for missing geometry",
    ],
    "Count": [
        len(requested_roads),
        len(mobility_contexts),
        len(geometry_exclusions),
    ],
}))

print("Records loaded. Identity consistency checks are next.")

,Item,Count
0,Road geometries supplied,700
1,Mobility road/direction contexts,1025
2,Roads excluded for missing geometry,3


Records loaded. Identity consistency checks are next.


In [5]:
requested_roads["mobility_segment_id"] = (
    requested_roads["mobility_segment_id"].astype(str)
)

assert requested_roads["mobility_segment_id"].notna().all()
assert mobility_contexts["mobility_unit_id"].notna().all()
assert mobility_contexts["mobility_segment_id"].notna().all()

assert not requested_roads["mobility_segment_id"].duplicated().any()
assert not mobility_contexts["mobility_unit_id"].duplicated().any()
assert not geometry_exclusions["mobility_segment_id"].duplicated().any()

road_ids = set(requested_roads["mobility_segment_id"])
excluded_ids = set(geometry_exclusions["mobility_segment_id"])

assert set(mobility_contexts["mobility_segment_id"]) == road_ids
assert road_ids.isdisjoint(excluded_ids)

actual_counts = mobility_contexts.groupby("mobility_segment_id").size()
declared_counts = pd.to_numeric(
    requested_roads.set_index("mobility_segment_id")["mobility_context_count"],
    errors="raise",
)

assert actual_counts.reindex(declared_counts.index).eq(declared_counts).all()

excluded_context_count = int(pd.to_numeric(
    geometry_exclusions["mobility_context_count"],
    errors="raise",
).sum())

assert excluded_context_count == 5
assert len(road_ids) + len(excluded_ids) == 703
assert len(mobility_contexts) + excluded_context_count == 1_030

print("Road and context identities reconcile.")
print("700 supplied roads: 1,025 contexts.")
print("3 geometry-excluded roads: 5 contexts.")
print("IDs are serving identities; current LION equivalence is not assumed.")
print("No geometry matching or training performed.")

Road and context identities reconcile.
700 supplied roads: 1,025 contexts.
3 geometry-excluded roads: 5 contexts.
IDs are serving identities; current LION equivalence is not assumed.
No geometry matching or training performed.


## 3. Scenario Time Window

Verify the New York calendar-year boundaries and hourly prediction starts.
The ending timestamp is excluded.

In [6]:
window = request_spec["window"]
timezone = window["timezone"]

start_local = pd.Timestamp(
    window["start_local_inclusive"]
).tz_convert(timezone)

end_local = pd.Timestamp(
    window["end_local_exclusive"]
).tz_convert(timezone)

assert start_local == pd.Timestamp("2024-01-01", tz=timezone)
assert end_local == pd.Timestamp("2025-01-01", tz=timezone)

START_UTC = start_local.tz_convert("UTC")
END_UTC = end_local.tz_convert("UTC")

assert START_UTC == pd.Timestamp(window["start_utc_inclusive"])
assert END_UTC == pd.Timestamp(window["end_utc_exclusive"])

requested_hours = pd.date_range(
    START_UTC, END_UTC, freq="h", inclusive="left"
)

assert len(requested_hours) == 8_784
assert len(requested_hours) == window[
    "expected_prediction_starts_per_supported_segment"
]

print("Local window:", start_local, "to", end_local)
print("UTC window:", START_UTC, "to", END_UTC)
print("Hourly prediction starts:", len(requested_hours))
print("Last prediction start:", requested_hours[-1])
print("End timestamp excluded; timezone changes handled.")
print("No inputs exported and no training performed.")

Local window: 2024-01-01 00:00:00-05:00 to 2025-01-01 00:00:00-05:00
UTC window: 2024-01-01 05:00:00+00:00 to 2025-01-01 05:00:00+00:00
Hourly prediction starts: 8784
Last prediction start: 2025-01-01 04:00:00+00:00
End timestamp excluded; timezone changes handled.
No inputs exported and no training performed.


## 4. Road Geometry Validation

Check coordinate systems and geometry quality before proposing native
climate matches. Preserve invalid or missing geometry as exclusions.

In [7]:
import geopandas as gpd

mobility_roads = gpd.read_file(
    REQUEST_DIR / "requested_mobility_roads.geojson"
)

if mobility_roads.crs is None:
    raise ValueError("The supplied geometries have no declared CRS.")

mobility_roads["mobility_segment_id"] = (
    mobility_roads["mobility_segment_id"].astype(str)
)

assert len(mobility_roads) == 700
assert set(mobility_roads["mobility_segment_id"]) == road_ids
assert not mobility_roads["mobility_segment_id"].duplicated().any()

METRIC_CRS = "EPSG:32618"
mobility_metric = mobility_roads.to_crs(METRIC_CRS)

geometry_checks = pd.DataFrame({
    "mobility_segment_id": mobility_metric["mobility_segment_id"],
    "missing_geometry": mobility_metric.geometry.isna(),
    "empty_geometry": mobility_metric.geometry.is_empty,
    "invalid_geometry": ~mobility_metric.geometry.is_valid,
    "unsupported_geometry_type": ~mobility_metric.geometry.geom_type.isin(
        ["LineString", "MultiLineString"]
    ),
    "nonpositive_length": mobility_metric.geometry.length.le(0),
})

check_columns = geometry_checks.columns.drop("mobility_segment_id")
geometry_checks["requires_review"] = (
    geometry_checks[check_columns].any(axis=1)
)

print("Supplied CRS:", mobility_roads.crs)
print("Distance/length assessment CRS:", METRIC_CRS)
print("Supplied roads checked:", len(geometry_checks))
print("Roads requiring geometry review:",
      int(geometry_checks["requires_review"].sum()))

if geometry_checks["requires_review"].any():
    display(
        geometry_checks.loc[geometry_checks["requires_review"]]
    )

print("The 3 previously excluded roads remain separately recorded.")
print("No geometry repaired, matches selected or files changed.")

Supplied CRS: EPSG:4326
Distance/length assessment CRS: EPSG:32618
Supplied roads checked: 700
Roads requiring geometry review: 0
The 3 previously excluded roads remain separately recorded.
No geometry repaired, matches selected or files changed.


In [8]:
climate_geometry_path = (
    PROJECT_ROOT
    / "data/processed/baseline_street_features_v1/street_segments.parquet"
)

climate_roads = gpd.read_parquet(climate_geometry_path)

if climate_roads.crs is None:
    raise ValueError("Native climate geometries have no CRS.")

if climate_roads["street_segment_id"].isna().any():
    raise ValueError("Missing native climate segment IDs.")

if climate_roads["street_segment_id"].duplicated().any():
    raise ValueError("Duplicate native climate segment IDs.")

climate_metric = climate_roads.to_crs(METRIC_CRS)

climate_geometry_usable = (
    climate_metric.geometry.notna()
    & ~climate_metric.geometry.is_empty
    & climate_metric.geometry.is_valid
    & climate_metric.geometry.geom_type.isin(
        ["LineString", "MultiLineString"]
    )
    & climate_metric.geometry.length.gt(0)
)

print("Native physical segments:", len(climate_metric))
print("Usable geometries:", int(climate_geometry_usable.sum()))
print("Unusable geometries:", int((~climate_geometry_usable).sum()))
print("Assessment CRS:", climate_metric.crs)
print("No road matches selected and no files changed.")

Native physical segments: 91895
Usable geometries: 91895
Unusable geometries: 0
Assessment CRS: EPSG:32618
No road matches selected and no files changed.


## 5. Geometry Candidate Assessment

Search within 20 metres and measure mutual coverage using a 10-metre
buffer. These settings produce mapping evidence, not automatic acceptance.

Retain alternatives and roads without candidates for review.

In [9]:
SEARCH_DISTANCE_M = 20
OVERLAP_BUFFER_M = 10
candidate_records = []

for road in mobility_metric.itertuples(index=False):
    nearby_indices = climate_metric.sindex.query(
        road.geometry.buffer(SEARCH_DISTANCE_M),
        predicate="intersects",
    )

    for native in climate_metric.iloc[nearby_indices].itertuples(index=False):
        candidate_records.append({
            "mobility_segment_id": road.mobility_segment_id,
            "lion_segment_id": str(road.lion_segment_id),
            "mobility_lion_key_used": road.mobility_lion_key_used,
            "supported_mobility_directions": road.supported_mobility_directions,
            "street_segment_id": native.street_segment_id,
            "minimum_distance_m": road.geometry.distance(native.geometry),
            "mobility_length_m": road.geometry.length,
            "climate_length_m": native.geometry.length,
            "mobility_covered_percent": (
                road.geometry.intersection(
                    native.geometry.buffer(OVERLAP_BUFFER_M)
                ).length / road.geometry.length * 100
            ),
            "climate_covered_percent": (
                native.geometry.intersection(
                    road.geometry.buffer(OVERLAP_BUFFER_M)
                ).length / native.geometry.length * 100
            ),
        })

if not candidate_records:
    raise ValueError("No nearby candidates found; review spatial alignment.")

mapping_candidates = pd.DataFrame(candidate_records)

roads_without_candidates = sorted(
    road_ids - set(mapping_candidates["mobility_segment_id"])
)

print("Candidate road-pair rows:", len(mapping_candidates))
print("Mobility roads with nearby candidates:",
      mapping_candidates["mobility_segment_id"].nunique())
print("Mobility roads without nearby candidates:",
      len(roads_without_candidates))
print("Search distance:", SEARCH_DISTANCE_M, "metres")
print("Overlap buffer:", OVERLAP_BUFFER_M, "metres")
print("No matches accepted; all candidate evidence retained in memory.")

Candidate road-pair rows: 4459
Mobility roads with nearby candidates: 699
Mobility roads without nearby candidates: 1
Search distance: 20 metres
Overlap buffer: 10 metres
No matches accepted; all candidate evidence retained in memory.


In [10]:
# A provisional inspection threshold, not mapping approval.
STRONG_OVERLAP_PERCENT = 95

mapping_candidates["strong_mutual_overlap"] = (
    mapping_candidates["mobility_covered_percent"].ge(
        STRONG_OVERLAP_PERCENT
    )
    & mapping_candidates["climate_covered_percent"].ge(
        STRONG_OVERLAP_PERCENT
    )
)

mapping_review = requested_roads[["mobility_segment_id"]].copy()

candidate_counts = mapping_candidates.groupby("mobility_segment_id").size()
strong_counts = mapping_candidates.loc[
    mapping_candidates["strong_mutual_overlap"]
].groupby("mobility_segment_id").size()

mapping_review["candidate_count"] = (
    mapping_review["mobility_segment_id"]
    .map(candidate_counts).fillna(0).astype(int)
)
mapping_review["strong_candidate_count"] = (
    mapping_review["mobility_segment_id"]
    .map(strong_counts).fillna(0).astype(int)
)

mapping_review["review_status"] = "partial_or_other_alignment_review"
mapping_review.loc[
    mapping_review["candidate_count"].eq(0), "review_status"
] = "no_candidate_within_search_distance"
mapping_review.loc[
    mapping_review["strong_candidate_count"].eq(1), "review_status"
] = "single_strong_candidate_pending_review"
mapping_review.loc[
    mapping_review["strong_candidate_count"].gt(1), "review_status"
] = "multiple_strong_candidates_review"

display(
    mapping_review.groupby("review_status")
    .size().rename("mobility_roads").to_frame()
)

print("Road without nearby candidates:")
display(
    requested_roads.loc[
        requested_roads["mobility_segment_id"].isin(
            roads_without_candidates
        )
    ]
)

assert len(mapping_review) == 700
print("All 700 requested roads accounted for.")
print("Partial and multiple matches remain unresolved, not discarded.")
print("No mapping accepted and no files changed.")

,mobility_roads
review_status,
multiple_strong_candidates_review,10
no_candidate_within_search_distance,1
partial_or_other_alignment_review,246
single_strong_candidate_pending_review,443


Road without nearby candidates:


,lion_segment_id,supported_mobility_directions,mobility_context_count,mobility_lion_key_used,mobility_segment_id
230,192731,WB,1,SegmentID,192731


All 700 requested roads accounted for.
Partial and multiple matches remain unresolved, not discarded.
No mapping accepted and no files changed.


## 6. Ambiguous Geometry Candidates

Inspect roads with multiple strong candidates. Retain every alternative;
do not select a segment using distance alone.

In [11]:
ambiguous_road_ids = mapping_review.loc[
    mapping_review["strong_candidate_count"].gt(1),
    "mobility_segment_id",
]

ambiguous_candidates = mapping_candidates.loc[
    mapping_candidates["mobility_segment_id"].isin(ambiguous_road_ids)
    & mapping_candidates["strong_mutual_overlap"]
].sort_values(
    ["mobility_segment_id", "minimum_distance_m", "street_segment_id"]
)

display(
    ambiguous_candidates[[
        "mobility_segment_id",
        "street_segment_id",
        "minimum_distance_m",
        "mobility_length_m",
        "climate_length_m",
        "mobility_covered_percent",
        "climate_covered_percent",
    ]].style.format({
        "minimum_distance_m": "{:.2f}",
        "mobility_length_m": "{:.2f}",
        "climate_length_m": "{:.2f}",
        "mobility_covered_percent": "{:.2f}",
        "climate_covered_percent": "{:.2f}",
    })
)

print("Ambiguous mobility roads:",
      ambiguous_candidates["mobility_segment_id"].nunique())
print("All strong alternatives retained; no mapping selected.")

,mobility_segment_id,street_segment_id,minimum_distance_m,mobility_length_m,climate_length_m,mobility_covered_percent,climate_covered_percent
546,146527,09aa2e0f1635232eda9525be9cde7dd092997f82b022b6ae8c50fb91708519fc,0.00,74.03,80.41,100.00,100.00
545,146527,dbe7eba765d1c41d4c10d4c2101dcdd4f50e238000e3db79b5ffdf6304913a87,7.54,74.03,80.56,100.00,96.32
871,152649,256ecc41d2b5d2e4105b032d228e4f3fef32423a80059252ba59580639dcfa2b,0.00,81.86,82.63,100.00,100.00
872,152649,5bdd0df254ada0a5b16966bd34a30a43916c9778fdc2696cf5a3649c489e55c4,9.39,81.86,82.68,100.00,100.00
896,152873,6015db199393d747b2afacf903a163e40dab330216acf80de8795b86ec337fc8,1.40,79.24,79.23,100.00,100.00
897,152873,4e12b3b78ad816a939ad60dc839730ff2242d94b4f598cd2ba9420867e6983a4,7.34,79.24,79.30,100.00,100.00
907,153067,4e12b3b78ad816a939ad60dc839730ff2242d94b4f598cd2ba9420867e6983a4,2.16,79.22,79.30,100.00,100.00
906,153067,6015db199393d747b2afacf903a163e40dab330216acf80de8795b86ec337fc8,8.08,79.22,79.23,100.00,100.00
995,155846,13d8e0ebf36421cc9f39db5c7cd5a92d8b8f0bfd92ec55980d6ce6184aa11f8e,1.24,147.37,144.48,100.00,100.00
990,155846,98fd0c4bbfa50cf3f8bb35a285b62cca0ec94e36c7b4d3aa9360b073f1354ff7,5.63,147.37,153.31,100.00,100.00


Ambiguous mobility roads: 10
All strong alternatives retained; no mapping selected.


In [12]:
partial_ids = mapping_review.loc[
    mapping_review["review_status"].eq(
        "partial_or_other_alignment_review"
    ),
    "mobility_segment_id",
]

partial_candidates = mapping_candidates.loc[
    mapping_candidates["mobility_segment_id"].isin(partial_ids)
].copy()

partial_candidates["alignment_evidence"] = "limited_mutual_overlap"

partial_candidates.loc[
    partial_candidates["climate_covered_percent"].ge(95),
    "alignment_evidence",
] = "native_segment_mostly_within_mobility_buffer"

partial_candidates.loc[
    partial_candidates["mobility_covered_percent"].ge(95),
    "alignment_evidence",
] = "mobility_road_mostly_within_native_buffer"

display(
    partial_candidates.groupby("alignment_evidence").agg(
        candidate_pairs=("street_segment_id", "size"),
        mobility_roads=("mobility_segment_id", "nunique"),
    )
)

print("Distinct partial-review roads:",
      partial_candidates["mobility_segment_id"].nunique())
print("Road counts across categories may overlap.")
print("These are review indicators, not accepted mapping types.")
print("No candidates discarded and no files changed.")

,candidate_pairs,mobility_roads
alignment_evidence,,
limited_mutual_overlap,893,218
mobility_road_mostly_within_native_buffer,242,223
native_segment_mostly_within_mobility_buffer,46,27


Distinct partial-review roads: 246
Road counts across categories may overlap.
These are review indicators, not accepted mapping types.
No candidates discarded and no files changed.


In [13]:
partial_summary = partial_candidates.groupby(
    "alignment_evidence"
).agg(
    candidate_pairs=("street_segment_id", "size"),
    mobility_roads=("mobility_segment_id", "nunique"),
)

print(partial_summary.to_string())

                                              candidate_pairs  mobility_roads
alignment_evidence                                                           
limited_mutual_overlap                                    893             218
mobility_road_mostly_within_native_buffer                 242             223
native_segment_mostly_within_mobility_buffer               46              27


In [14]:
from shapely.ops import unary_union

fragment_candidates = partial_candidates.loc[
    partial_candidates["climate_covered_percent"].ge(95)
]

mobility_lookup = mobility_metric.set_index("mobility_segment_id")
climate_lookup = climate_metric.set_index("street_segment_id")
fragment_review_rows = []

for mobility_id, candidates in fragment_candidates.groupby(
    "mobility_segment_id"
):
    mobility_geometry = mobility_lookup.loc[mobility_id, "geometry"]

    native_geometries = climate_lookup.loc[
        candidates["street_segment_id"].unique(), "geometry"
    ]

    combined_buffer = unary_union([
        geometry.buffer(OVERLAP_BUFFER_M)
        for geometry in native_geometries
    ])

    fragment_review_rows.append({
        "mobility_segment_id": mobility_id,
        "native_candidate_count": len(native_geometries),
        "combined_mobility_covered_percent": (
            mobility_geometry.intersection(combined_buffer).length
            / mobility_geometry.length * 100
        ),
    })

fragment_review = pd.DataFrame(fragment_review_rows)

print(
    fragment_review.sort_values(
        "combined_mobility_covered_percent", ascending=False
    ).to_string(index=False, float_format=lambda value: f"{value:.2f}")
)

print("\nCombined coverage is buffered geometry evidence only.")
print("Connectivity, competing alignments and mapping approval remain unresolved.")
print("No native segments merged and no files changed.")

mobility_segment_id  native_candidate_count  combined_mobility_covered_percent
              91014                       2                             100.00
             113476                       4                             100.00
              79377                       2                             100.00
              78177                       3                             100.00
              70704                       3                             100.00
              33018                       4                             100.00
               4297                       1                              91.43
             116598                       1                              88.93
              57192                       1                              86.05
               8070                       1                              80.79
             191274                       1                              76.33
             174360                       2         

In [15]:
edge_bridge = pd.read_parquet(
    PROJECT_ROOT / "data/processed/physical_street_index_v1.parquet",
    columns=["street_segment_id", "u", "v"],
)

potential_split_ids = fragment_review.loc[
    fragment_review["native_candidate_count"].gt(1)
    & fragment_review["combined_mobility_covered_percent"].ge(95),
    "mobility_segment_id",
]

connectivity_rows = []

for mobility_id in potential_split_ids:
    segment_ids = set(fragment_candidates.loc[
        fragment_candidates["mobility_segment_id"].eq(mobility_id),
        "street_segment_id",
    ])

    nodes_by_segment = {}
    for segment_id in segment_ids:
        edges = edge_bridge.loc[
            edge_bridge["street_segment_id"].eq(segment_id)
        ]
        if edges.empty:
            raise ValueError(f"Missing OSM bridge: {segment_id}")
        nodes_by_segment[segment_id] = set(
            edges[["u", "v"]].to_numpy().ravel()
        )

    remaining = set(segment_ids)
    components = 0

    while remaining:
        components += 1
        pending = [remaining.pop()]

        while pending:
            current = pending.pop()
            neighbours = {
                other for other in remaining
                if nodes_by_segment[current] & nodes_by_segment[other]
            }
            remaining -= neighbours
            pending.extend(neighbours)

    connectivity_rows.append({
        "mobility_segment_id": mobility_id,
        "native_segments": len(segment_ids),
        "connected_components": components,
        "shared_node_connected": components == 1,
    })

split_connectivity = pd.DataFrame(connectivity_rows)
print(split_connectivity.to_string(index=False))

print("\nConnectivity uses shared OSM endpoint identities.")
print("It does not verify traffic direction or approve the mapping.")

mobility_segment_id  native_segments  connected_components  shared_node_connected
             113476                4                     1                   True
              33018                4                     1                   True
              70704                3                     1                   True
              78177                3                     1                   True
              79377                2                     1                   True
              91014                2                     1                   True

Connectivity uses shared OSM endpoint identities.
It does not verify traffic direction or approve the mapping.


In [16]:
single_match_ids = mapping_review.loc[
    mapping_review["review_status"].eq(
        "single_strong_candidate_pending_review"
    ),
    "mobility_segment_id",
]

split_match_ids = split_connectivity.loc[
    split_connectivity["shared_node_connected"],
    "mobility_segment_id",
]

draft_crosswalk = mapping_candidates.loc[
    (
        mapping_candidates["mobility_segment_id"].isin(single_match_ids)
        & mapping_candidates["strong_mutual_overlap"]
    )
    | (
        mapping_candidates["mobility_segment_id"].isin(split_match_ids)
        & mapping_candidates["climate_covered_percent"].ge(95)
    )
].copy()

draft_crosswalk["mapping_status"] = "proposed_pending_review"
draft_crosswalk["relationship"] = "single_native_candidate"

draft_crosswalk.loc[
    draft_crosswalk["mobility_segment_id"].isin(split_match_ids),
    "relationship",
] = "connected_native_segments"

native_reuse_counts = draft_crosswalk.groupby(
    "street_segment_id"
)["mobility_segment_id"].nunique()

draft_crosswalk["native_shared_between_mobility_roads"] = (
    draft_crosswalk["street_segment_id"]
    .map(native_reuse_counts).gt(1)
)

display(
    draft_crosswalk.groupby("relationship").agg(
        mobility_roads=("mobility_segment_id", "nunique"),
        mapping_pairs=("street_segment_id", "size"),
        native_segments=("street_segment_id", "nunique"),
    )
)

shared_native = draft_crosswalk.loc[
    draft_crosswalk["native_shared_between_mobility_roads"]
]

print("Native segments shared between mobility roads:",
      shared_native["street_segment_id"].nunique())

if not shared_native.empty:
    display(shared_native[[
        "mobility_segment_id", "street_segment_id", "relationship"
    ]])

print("Requested roads without a draft relationship:",
      len(road_ids - set(draft_crosswalk["mobility_segment_id"])))
print("All alternatives remain preserved in mapping_candidates.")
print("No mapping approved and no files changed.")

,mobility_roads,mapping_pairs,native_segments
relationship,,,
connected_native_segments,6,18,18
single_native_candidate,443,443,442


Native segments shared between mobility roads: 1


,mobility_segment_id,street_segment_id,relationship
485,145550,6945d8a240609b3a89579415f2792e20b5c5f31a239f33...,single_native_candidate
3486,51124,6945d8a240609b3a89579415f2792e20b5c5f31a239f33...,single_native_candidate


Requested roads without a draft relationship: 251
All alternatives remain preserved in mapping_candidates.
No mapping approved and no files changed.


## 7. Native Segment Input Support

Verify static features and saved road/radar identities for the draft
crosswalk. Mapping acceptance and the new model release remain pending.

In [17]:
import numpy as np

static_features = pd.read_parquet(
    PROJECT_ROOT
    / "data/processed/baseline_street_features_v1/street_features.parquet"
)

release_config = json.loads(
    (PROJECT_ROOT / "config/climate_history_release_v3_02.json").read_text()
)
reference_bridge = pd.read_parquet(
    PROJECT_ROOT / release_config["bundle_path"]
    / "street_cell_bridge.parquet"
)

for frame in [static_features, reference_bridge]:
    assert not frame["street_segment_id"].duplicated().any()

native_ids = draft_crosswalk[["street_segment_id"]].drop_duplicates()

native_support = native_ids.merge(
    static_features, on="street_segment_id",
    how="left", validate="one_to_one",
).merge(
    reference_bridge[["street_segment_id", "grid_id", "cell_id"]].rename(
        columns={"grid_id": "reference_grid_id",
                 "cell_id": "reference_cell_id"}
    ),
    on="street_segment_id", how="left", validate="one_to_one",
)

terrain_columns = [
    "elevation_mean_m", "terrain_endpoint_grade_abs",
    "terrain_relief_m", "length_m",
]

native_support["static_inputs_valid"] = (
    native_support["static_features_valid"].eq(True)
    & np.isfinite(
        native_support[terrain_columns].to_numpy(dtype=float)
    ).all(axis=1)
    & native_support["length_m"].gt(0)
)

native_support["reference_bridge_matches"] = (
    native_support["grid_id"].eq(native_support["reference_grid_id"])
    & native_support["cell_id"].eq(native_support["reference_cell_id"])
)

display(pd.DataFrame({
    "Check": [
        "Distinct proposed native segments",
        "Valid static inputs",
        "Matching reference bridge identities",
    ],
    "Count": [
        len(native_support),
        int(native_support["static_inputs_valid"].sum()),
        int(native_support["reference_bridge_matches"].sum()),
    ],
}))

unsupported_native = native_support.loc[
    ~native_support["static_inputs_valid"]
    | ~native_support["reference_bridge_matches"]
]

if not unsupported_native.empty:
    display(unsupported_native[[
        "street_segment_id", "static_inputs_valid",
        "reference_bridge_matches",
    ]])

print("No model loaded, training performed or files changed.")

,Check,Count
0,Distinct proposed native segments,460
1,Valid static inputs,460
2,Matching reference bridge identities,460


No model loaded, training performed or files changed.


## 8. Exact-Window Rainfall Coverage

Assess rainfall inputs for all 8,784 requested prediction hours.
Include adjoining UTC partitions and retain unavailable inputs explicitly.

In [18]:
first_month = START_UTC.tz_localize(None).to_period("M")
last_month = (
    END_UTC - pd.Timedelta(hours=1)
).tz_localize(None).to_period("M")

weather_partitions = []

for month in pd.period_range(first_month, last_month, freq="M"):
    directory = (
        PROJECT_ROOT / "data/processed/baseline_weather_v1" / str(month)
    )
    weather_partitions.append({
        "month_utc": str(month),
        "feature_path": directory / "weather_features.parquet",
        "manifest_path": directory / "manifest.json",
        "features_present": (directory / "weather_features.parquet").is_file(),
        "manifest_present": (directory / "manifest.json").is_file(),
    })

partition_inventory = pd.DataFrame(weather_partitions)

display(partition_inventory[[
    "month_utc", "features_present", "manifest_present"
]])

print("Required UTC partitions:", len(partition_inventory))
print("File presence only; hourly coverage is not yet verified.")

,month_utc,features_present,manifest_present
0,2024-01,True,True
1,2024-02,True,True
2,2024-03,True,True
3,2024-04,True,True
4,2024-05,True,True
5,2024-06,True,True
6,2024-07,True,True
7,2024-08,True,True
8,2024-09,True,True
9,2024-10,True,True


Required UTC partitions: 13
File presence only; hourly coverage is not yet verified.


In [19]:
selected_cells = native_support[
    ["grid_id", "cell_id"]
].drop_duplicates()

assert selected_cells["grid_id"].nunique() == 1
selected_grid = selected_cells["grid_id"].iloc[0]
selected_cell_ids = selected_cells["cell_id"].tolist()

rainfall_columns = [
    "grid_id", "cell_id", "prediction_time_utc",
    "rain_1h_mm", "rain_3h_mm", "rain_6h_mm", "rain_24h_mm",
    "weather_features_valid",
]

rainfall_parts = []

for partition in weather_partitions:
    rainfall_parts.append(pd.read_parquet(
        partition["feature_path"],
        columns=rainfall_columns,
        filters=[
            ("grid_id", "==", selected_grid),
            ("cell_id", "in", selected_cell_ids),
            ("prediction_time_utc", ">=", START_UTC.to_pydatetime()),
            ("prediction_time_utc", "<", END_UTC.to_pydatetime()),
        ],
    ))

scenario_rainfall = pd.concat(rainfall_parts, ignore_index=True)

rainfall_keys = ["grid_id", "cell_id", "prediction_time_utc"]
assert not scenario_rainfall.duplicated(rainfall_keys).any()

print("Distinct requested radar cells:", len(selected_cells))
print("Loaded cell-hour rows:", len(scenario_rainfall))
print("Expected cell-hour rows:", len(selected_cells) * len(requested_hours))
print("Observed prediction hours:",
      scenario_rainfall["prediction_time_utc"].nunique())
print("No duplicate cell-hour keys.")
print("Feature validity and absent rows will be checked next.")

Distinct requested radar cells: 237
Loaded cell-hour rows: 2081808
Expected cell-hour rows: 2081808
Observed prediction hours: 8784
No duplicate cell-hour keys.
Feature validity and absent rows will be checked next.


In [20]:
rain_features = [
    "rain_1h_mm", "rain_3h_mm", "rain_6h_mm", "rain_24h_mm"
]

assert scenario_rainfall["prediction_time_utc"].isin(requested_hours).all()

hours_per_cell = scenario_rainfall.groupby(
    ["grid_id", "cell_id"]
).size()
assert hours_per_cell.eq(len(requested_hours)).all()
assert len(hours_per_cell) == len(selected_cells)

rain_values = scenario_rainfall[rain_features].to_numpy(dtype=float)

scenario_rainfall["rainfall_inputs_valid"] = (
    scenario_rainfall["weather_features_valid"].eq(True)
    & np.isfinite(rain_values).all(axis=1)
    & (rain_values >= 0).all(axis=1)
    & (np.diff(rain_values, axis=1) >= -1e-6).all(axis=1)
)

unavailable_cell_hours = scenario_rainfall.loc[
    ~scenario_rainfall["rainfall_inputs_valid"],
    rainfall_keys,
].copy()

unavailable_hour_summary = (
    unavailable_cell_hours.groupby("prediction_time_utc")
    .size()
    .rename("unavailable_cells")
    .reset_index()
)

print("Valid cell-hours:",
      int(scenario_rainfall["rainfall_inputs_valid"].sum()))
print("Unavailable cell-hours:", len(unavailable_cell_hours))
print("Prediction hours with any unavailable cells:",
      len(unavailable_hour_summary))

if not unavailable_hour_summary.empty:
    print("First affected hour:",
          unavailable_hour_summary["prediction_time_utc"].min())
    print("Last affected hour:",
          unavailable_hour_summary["prediction_time_utc"].max())
    print("Unavailable cells per affected hour:",
          sorted(unavailable_hour_summary["unavailable_cells"].unique()))

print("Affected timestamps above are UTC and inclusive.")
print("Unavailable inputs retained; no rainfall filled with zero.")
print("No predictions generated and no files changed.")

Valid cell-hours: 2075646
Unavailable cell-hours: 6162
Prediction hours with any unavailable cells: 26
First affected hour: 2024-03-19 20:00:00+00:00
Last affected hour: 2024-03-20 21:00:00+00:00
Unavailable cells per affected hour: [np.int64(237)]
Affected timestamps above are UTC and inclusive.
Unavailable inputs retained; no rainfall filled with zero.
No predictions generated and no files changed.


## 9. Monthly Rainfall Coverage

Summarize the exact scenario window by New York calendar month.
An hour is fully usable when all selected radar cells have valid inputs.

In [21]:
hourly_coverage = scenario_rainfall.groupby(
    "prediction_time_utc"
).agg(
    cell_hours=("cell_id", "size"),
    valid_cell_hours=("rainfall_inputs_valid", "sum"),
)

hourly_coverage["fully_usable"] = (
    hourly_coverage["cell_hours"].eq(len(selected_cells))
    & hourly_coverage["valid_cell_hours"].eq(len(selected_cells))
)

hourly_coverage["month_new_york"] = (
    hourly_coverage.index
    .tz_convert("America/New_York")
    .strftime("%Y-%m")
)

monthly_input_coverage = hourly_coverage.groupby(
    "month_new_york"
).agg(
    expected_prediction_hours=("cell_hours", "size"),
    fully_usable_prediction_hours=("fully_usable", "sum"),
    total_cell_hours=("cell_hours", "sum"),
    valid_cell_hours=("valid_cell_hours", "sum"),
)

monthly_input_coverage["unavailable_prediction_hours"] = (
    monthly_input_coverage["expected_prediction_hours"]
    - monthly_input_coverage["fully_usable_prediction_hours"]
)

assert monthly_input_coverage["expected_prediction_hours"].sum() == 8_784
assert len(monthly_input_coverage) == 12

display(monthly_input_coverage)

print("Months use America/New_York boundaries, including DST changes.")
print("Coverage applies to the 237 cells supporting the draft mappings.")
print("Road mappings remain proposed; no files changed.")

,expected_prediction_hours,fully_usable_prediction_hours,total_cell_hours,valid_cell_hours,unavailable_prediction_hours
month_new_york,,,,,
2024-01,744,744,176328,176328,0
2024-02,696,696,164952,164952,0
2024-03,743,717,176091,169929,26
2024-04,720,720,170640,170640,0
2024-05,744,744,176328,176328,0
2024-06,720,720,170640,170640,0
2024-07,744,744,176328,176328,0
2024-08,744,744,176328,176328,0
2024-09,720,720,170640,170640,0


Months use America/New_York boundaries, including DST changes.
Coverage applies to the 237 cells supporting the draft mappings.
Road mappings remain proposed; no files changed.


## 10. Boundary Rainfall Source Verification

Verify the 24 contributing hourly sources for the first and last
prediction starts, retaining the existing three-hour rainfall lag.

In [22]:
boundary_source_rows = []

boundaries = [
    ("first_prediction", START_UTC, "2024-01"),
    ("last_prediction", requested_hours[-1], "2025-01"),
]

for boundary, prediction_time, month in boundaries:
    manifest_path = (
        PROJECT_ROOT / "data/processed/baseline_weather_v1"
        / month / "manifest.json"
    )
    weather_manifest = json.loads(manifest_path.read_text())

    assert weather_manifest["source_grid_id"] == selected_grid
    assert weather_manifest["preparation_config"][
        "rainfall_availability_lag_hours"
    ] == 3

    latest_end = prediction_time - pd.Timedelta(hours=3)
    source_ends = pd.date_range(
        latest_end - pd.Timedelta(hours=23),
        latest_end,
        freq="h",
    )

    for source_end in source_ends:
        filename = f"radar_{source_end:%Y%m%d_%H%M%S}.parquet"
        source_path = (
            PROJECT_ROOT / "data/raw/radar_parquet_v2" / filename
        )
        expected_hash = weather_manifest["source_sha256"].get(filename)

        if not source_path.is_file() or expected_hash is None:
            raise ValueError(f"Boundary source unavailable: {filename}")

        actual_hash = hashlib.sha256(source_path.read_bytes()).hexdigest()
        if actual_hash != expected_hash:
            raise ValueError(f"Boundary source hash mismatch: {filename}")

        boundary_source_rows.append({
            "boundary": boundary,
            "prediction_time_utc": prediction_time,
            "interval_start_utc": source_end - pd.Timedelta(hours=1),
            "interval_end_utc": source_end,
            "source_file": filename,
            "sha256": actual_hash,
        })

boundary_source_ledger = pd.DataFrame(boundary_source_rows)

display(
    boundary_source_ledger.groupby("boundary").agg(
        verified_source_hours=("source_file", "size"),
        first_interval_start_utc=("interval_start_utc", "min"),
        last_interval_end_utc=("interval_end_utc", "max"),
    )
)

assert len(boundary_source_ledger) == 48
print("Both boundary lookbacks have 24 hash-verified hourly sources.")
print("Actual historic publication times remain unverified.")
print("No files changed and no predictions generated.")

,verified_source_hours,first_interval_start_utc,last_interval_end_utc
boundary,,,
first_prediction,24,2023-12-31 02:00:00+00:00,2024-01-01 02:00:00+00:00
last_prediction,24,2024-12-31 01:00:00+00:00,2025-01-01 01:00:00+00:00


Both boundary lookbacks have 24 hash-verified hourly sources.
Actual historic publication times remain unverified.
No files changed and no predictions generated.


In [23]:
recorded_source_hashes = {}
recorded_missing_hours = set()
weather_manifest_hashes = {}

for partition in weather_partitions:
    path = partition["manifest_path"]
    content = path.read_bytes()
    manifest = json.loads(content)

    weather_manifest_hashes[str(path)] = (
        hashlib.sha256(content).hexdigest()
    )

    for filename, digest in manifest["source_sha256"].items():
        previous = recorded_source_hashes.get(filename)
        if previous is not None and previous != digest:
            raise ValueError(f"Conflicting source hashes: {filename}")
        recorded_source_hashes[filename] = digest

    recorded_missing_hours.update(
        pd.to_datetime(manifest["missing_source_hours"], utc=True)
    )

required_source_ends = pd.date_range(
    START_UTC - pd.Timedelta(hours=26),
    requested_hours[-1] - pd.Timedelta(hours=3),
    freq="h",
)

required_missing_hours = [
    timestamp for timestamp in required_source_ends
    if timestamp in recorded_missing_hours
]

print("Required raw source hours:", len(required_source_ends))
print("First source interval end:", required_source_ends[0])
print("Last source interval end:", required_source_ends[-1])
print("Declared missing source hours:", len(required_missing_hours))

for timestamp in required_missing_hours:
    print("Missing:", timestamp)

print("Source inventory only; full-year file hashes are checked next.")

Required raw source hours: 8807
First source interval end: 2023-12-31 03:00:00+00:00
Last source interval end: 2025-01-01 01:00:00+00:00
Declared missing source hours: 3
Missing: 2024-03-19 17:00:00+00:00
Missing: 2024-03-19 18:00:00+00:00
Missing: 2024-03-19 19:00:00+00:00
Source inventory only; full-year file hashes are checked next.


In [24]:
source_verification_rows = []

for position, timestamp in enumerate(required_source_ends, start=1):
    filename = f"radar_{timestamp:%Y%m%d_%H%M%S}.parquet"
    path = PROJECT_ROOT / "data/raw/radar_parquet_v2" / filename
    expected_hash = recorded_source_hashes.get(filename)

    if timestamp in recorded_missing_hours:
        if path.exists() or expected_hash is not None:
            raise ValueError(f"Declared gap needs investigation: {filename}")
        status = "declared_missing"
        actual_hash = None
    else:
        if not path.is_file() or expected_hash is None:
            raise ValueError(f"Unrecorded source gap: {filename}")

        actual_hash = hashlib.sha256(path.read_bytes()).hexdigest()
        if actual_hash != expected_hash:
            raise ValueError(f"Source hash mismatch: {filename}")
        status = "verified"

    source_verification_rows.append({
        "source_file": filename,
        "interval_start_utc": timestamp - pd.Timedelta(hours=1),
        "interval_end_utc": timestamp,
        "status": status,
        "sha256": actual_hash,
    })

    if position % 1000 == 0:
        print(f"Checked {position:,}/{len(required_source_ends):,} hours")

rainfall_source_ledger = pd.DataFrame(source_verification_rows)

display(
    rainfall_source_ledger.groupby("status")
    .size().rename("source_hours").to_frame()
)

assert len(rainfall_source_ledger) == 8_807
assert rainfall_source_ledger["status"].eq("declared_missing").sum() == 3

print("Full-window raw-source verification complete.")
print("Declared gaps retained; no inputs modified.")

Checked 1,000/8,807 hours
Checked 2,000/8,807 hours
Checked 3,000/8,807 hours
Checked 4,000/8,807 hours
Checked 5,000/8,807 hours
Checked 6,000/8,807 hours
Checked 7,000/8,807 hours
Checked 8,000/8,807 hours


,source_hours
status,
declared_missing,3
verified,8804


Full-window raw-source verification complete.
Declared gaps retained; no inputs modified.


In [25]:
expected_gap_hours = set()

for source_end in required_missing_hours:
    expected_gap_hours.update(pd.date_range(
        source_end + pd.Timedelta(hours=3),
        source_end + pd.Timedelta(hours=26),
        freq="h",
    ))

expected_gap_hours &= set(requested_hours)
observed_gap_hours = set(
    unavailable_hour_summary["prediction_time_utc"]
)

assert expected_gap_hours == observed_gap_hours

reported_gap = request_spec["known_reported_gap"]
reported_gap_hours = set(pd.date_range(
    reported_gap["prediction_start_utc_inclusive"],
    reported_gap["prediction_end_utc_exclusive"],
    freq="h",
    inclusive="left",
))

assert observed_gap_hours == reported_gap_hours

unavailable_prediction_hours = unavailable_hour_summary.copy()
unavailable_prediction_hours["prediction_time_new_york"] = (
    unavailable_prediction_hours["prediction_time_utc"]
    .dt.tz_convert("America/New_York")
)
unavailable_prediction_hours["reason"] = (
    "missing_source_within_required_rainfall_window"
)

print("Source-derived unavailable hours:", len(expected_gap_hours))
print("Observed unavailable hours:", len(observed_gap_hours))
print("Exact timestamps match the reported March gap.")
print("Exact-hour ledger prepared in memory; no files changed.")

Source-derived unavailable hours: 26
Observed unavailable hours: 26
Exact timestamps match the reported March gap.
Exact-hour ledger prepared in memory; no files changed.


## 11. Proposed Pre-2024 Time Policy

Use chronological training and development periods.
Require each target hour to finish at least seven days before its
relevant availability cutoff.

Retain training-only history; development and 2024 reports do not
enter the frozen historical-event pool.

In [26]:
REPORT_DELAY = pd.Timedelta(days=7)
MODEL_FREEZE_UTC = START_UTC

TRAIN_START_UTC = pd.Timestamp("2020-10-14T04:00:00Z")
TRAIN_END_UTC = pd.Timestamp("2022-12-25T05:00:00Z")
DEV_START_UTC = pd.Timestamp("2023-01-01T05:00:00Z")
DEV_END_UTC = MODEL_FREEZE_UTC - REPORT_DELAY

assert TRAIN_START_UTC < TRAIN_END_UTC < DEV_START_UTC < DEV_END_UTC
assert TRAIN_END_UTC + REPORT_DELAY <= DEV_START_UTC
assert DEV_END_UTC + REPORT_DELAY <= MODEL_FREEZE_UTC

proposed_periods = pd.DataFrame([
    {
        "partition": "training",
        "start_utc_inclusive": TRAIN_START_UTC,
        "end_utc_exclusive": TRAIN_END_UTC,
        "label_availability_cutoff": DEV_START_UTC,
    },
    {
        "partition": "development",
        "start_utc_inclusive": DEV_START_UTC,
        "end_utc_exclusive": DEV_END_UTC,
        "label_availability_cutoff": MODEL_FREEZE_UTC,
    },
])

display(proposed_periods)

print("Model freeze:", MODEL_FREEZE_UTC)
print("Label rule: target_end + 7 days <= availability cutoff.")
print("Availability delay remains assumed, not historically verified.")
print("History would use training-period events only.")
print("No configuration saved and no model fitted.")

,partition,start_utc_inclusive,end_utc_exclusive,label_availability_cutoff
0,training,2020-10-14 04:00:00+00:00,2022-12-25 05:00:00+00:00,2023-01-01 05:00:00+00:00
1,development,2023-01-01 05:00:00+00:00,2023-12-25 05:00:00+00:00,2024-01-01 05:00:00+00:00


Model freeze: 2024-01-01 05:00:00+00:00
Label rule: target_end + 7 days <= availability cutoff.
Availability delay remains assumed, not historically verified.
History would use training-period events only.
No configuration saved and no model fitted.


In [27]:
import duckdb

database_path = (
    PROJECT_ROOT / "data/processed/baseline_dataset_v1/dataset.duckdb"
)

with duckdb.connect(str(database_path), read_only=True) as connection:
    connection.execute("SET threads=2")
    connection.execute("SET TimeZone='UTC'")

    proposed_history_events = connection.execute("""
        SELECT
            e.street_segment_id,
            s.grid_id,
            s.cell_id,
            e.prediction_time_utc
        FROM eligible AS e
        JOIN street_features AS s USING (street_segment_id)
        WHERE e.label = 1
          AND e.prediction_time_utc >= ?::TIMESTAMPTZ
          AND e.prediction_time_utc < ?::TIMESTAMPTZ
          AND e.prediction_time_utc
              + INTERVAL '1 hour'
              + INTERVAL '7 days' <= ?::TIMESTAMPTZ
        ORDER BY e.street_segment_id, e.prediction_time_utc
    """, [
        TRAIN_START_UTC.isoformat(),
        TRAIN_END_UTC.isoformat(),
        DEV_START_UTC.isoformat(),
    ]).df()

assert not proposed_history_events.empty
assert not proposed_history_events.duplicated(
    ["street_segment_id", "prediction_time_utc"]
).any()

available_by = (
    proposed_history_events["prediction_time_utc"]
    + pd.Timedelta(hours=1) + REPORT_DELAY
)
assert available_by.le(DEV_START_UTC).all()
assert proposed_history_events["prediction_time_utc"].lt(TRAIN_END_UTC).all()

print("Eligible positive segment-hours:", len(proposed_history_events))
print("First event:", proposed_history_events["prediction_time_utc"].min())
print("Last event:", proposed_history_events["prediction_time_utc"].max())
print("All events meet the proposed training availability cutoff.")
print("No development or 2024 events included.")
print("Proposed history events held in memory; no model fitted.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Eligible positive segment-hours: 6739
First event: 2020-10-14 21:00:00+00:00
Last event: 2022-12-25 03:00:00+00:00
All events meet the proposed training availability cutoff.
No development or 2024 events included.
Proposed history events held in memory; no model fitted.


In [28]:
import sys

scripts_path = str(PROJECT_ROOT / "scripts")
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)

from climate_history_features import HistoricalReportFeatures

proposed_history_builder = HistoricalReportFeatures(
    proposed_history_events,
    training_start=TRAIN_START_UTC,
    training_end=TRAIN_END_UTC,
    lookback_days=365,
    delay_days=7,
)

history_check_rows = native_support[
    ["street_segment_id", "grid_id", "cell_id"]
].merge(
    pd.DataFrame({
        "prediction_time_utc": [requested_hours[0], requested_hours[-1]]
    }),
    how="cross",
)

history_features = proposed_history_builder.transform(history_check_rows)
history_support_check = pd.concat(
    [history_check_rows, history_features], axis=1
)

# The existing policy clamps the cutoff to training_end throughout 2024.
assert ((requested_hours - REPORT_DELAY) >= TRAIN_END_UTC).all()
assert history_features["history_cutoff_utc"].eq(TRAIN_END_UTC).all()

history_columns = [
    "segment_positive_hours_365d",
    "cell_positive_hours_365d",
]

assert np.isfinite(history_features[history_columns].to_numpy()).all()
assert history_features[history_columns].ge(0).all().all()
assert history_support_check.groupby(
    "street_segment_id"
)[history_columns].nunique().eq(1).all().all()

print("Native segments checked:",
      history_check_rows["street_segment_id"].nunique())
print("Boundary rows checked:", len(history_check_rows))
print("Frozen history cutoff:", TRAIN_END_UTC)
print("History construction is executable across the 2024 window.")
print("No development or 2024 reports enter the event pool.")
print("Zero history counts are valid; they do not indicate missing inputs.")
print("No new model fitted and no files changed.")

Native segments checked: 460
Boundary rows checked: 920
Frozen history cutoff: 2022-12-25 05:00:00+00:00
History construction is executable across the 2024 window.
No development or 2024 reports enter the event pool.
Zero history counts are valid; they do not indicate missing inputs.
No new model fitted and no files changed.


In [29]:
road_assessment = requested_roads.merge(
    mapping_review,
    on="mobility_segment_id",
    validate="one_to_one",
)

draft_counts = draft_crosswalk.groupby(
    "mobility_segment_id"
)["street_segment_id"].nunique()

shared_flags = draft_crosswalk.groupby(
    "mobility_segment_id"
)["native_shared_between_mobility_roads"].any()

road_assessment["proposed_native_segments"] = (
    road_assessment["mobility_segment_id"]
    .map(draft_counts).fillna(0).astype(int)
)

road_assessment["shared_native_review_required"] = (
    road_assessment["mobility_segment_id"]
    .map(shared_flags).eq(True)
)

road_assessment["assessment_status"] = "mapping_review_pending"
road_assessment.loc[
    road_assessment["proposed_native_segments"].gt(0),
    "assessment_status",
] = "proposed_mapping_with_input_checks"

road_assessment["mobility_context_count"] = pd.to_numeric(
    road_assessment["mobility_context_count"], errors="raise"
)

assert len(road_assessment) == 700
assert road_assessment["mobility_context_count"].sum() == 1_025

display(
    road_assessment.groupby("assessment_status").agg(
        physical_roads=("mobility_segment_id", "size"),
        mobility_contexts=("mobility_context_count", "sum"),
    )
)

print("Roads requiring shared-native review:",
      int(road_assessment["shared_native_review_required"].sum()))
print("The 3 supplied geometry exclusions remain separately recorded.")
print("Unresolved mappings are pending review, not declared unsupported.")
print("No mapping approved and no files changed.")

,physical_roads,mobility_contexts
assessment_status,,
mapping_review_pending,251,347
proposed_mapping_with_input_checks,449,678


Roads requiring shared-native review: 2
The 3 supplied geometry exclusions remain separately recorded.
Unresolved mappings are pending review, not declared unsupported.
No mapping approved and no files changed.


In [30]:
ASSESSMENT_DIR = (
    PROJECT_ROOT
    / "data/processed/climate_coverage_assessments"
    / "pre2024-mobility-assessment-v1"
)

assert len(road_assessment) == 700
assert len(draft_crosswalk) == 461
assert draft_crosswalk["street_segment_id"].nunique() == 460

# Refuse to overwrite an existing assessment.
ASSESSMENT_DIR.mkdir(parents=True, exist_ok=False)

assessment_tables = {
    "road_assessment.csv": road_assessment,
    "proposed_mobility_crosswalk.csv": draft_crosswalk,
    "mapping_candidates.csv": mapping_candidates,
    "mapping_review_pending.csv": road_assessment.loc[
        road_assessment["assessment_status"].eq("mapping_review_pending")
    ],
    "geometry_exclusions.csv": geometry_exclusions,
    "monthly_input_coverage.csv": monthly_input_coverage.reset_index(),
    "unavailable_prediction_hours.csv": unavailable_prediction_hours,
    "unavailable_cell_hours.csv": unavailable_cell_hours,
    "rainfall_source_ledger.csv": rainfall_source_ledger,
    "boundary_source_ledger.csv": boundary_source_ledger,
}

for filename, table in assessment_tables.items():
    table.to_csv(ASSESSMENT_DIR / filename, index=False)

display(pd.DataFrame([
    {"file": filename, "rows": len(table)}
    for filename, table in assessment_tables.items()
]))

print("Assessment folder:", ASSESSMENT_DIR)
print("Tables saved. Geometry, provenance and documentation are next.")
print("No model or existing release changed.")

,file,rows
0,road_assessment.csv,700
1,proposed_mobility_crosswalk.csv,461
2,mapping_candidates.csv,4459
3,mapping_review_pending.csv,251
4,geometry_exclusions.csv,3
5,monthly_input_coverage.csv,12
6,unavailable_prediction_hours.csv,26
7,unavailable_cell_hours.csv,6162
8,rainfall_source_ledger.csv,8807
9,boundary_source_ledger.csv,48


Assessment folder: /Users/macbookairm3/Documents/Plexa-A-Multi-Modal-Smart-City-Digital-Twin-and-Cascading-Resilience-Engine/data/processed/climate_coverage_assessments/pre2024-mobility-assessment-v1
Tables saved. Geometry, provenance and documentation are next.
No model or existing release changed.


In [31]:
proposed_native_ids = set(draft_crosswalk["street_segment_id"])

assessment_geometry = climate_roads.loc[
    climate_roads["street_segment_id"].isin(proposed_native_ids)
].copy()

full_edge_bridge = pd.read_parquet(
    PROJECT_ROOT / "data/processed/physical_street_index_v1.parquet"
)
assessment_edges = full_edge_bridge.loc[
    full_edge_bridge["street_segment_id"].isin(proposed_native_ids)
].copy()

assert set(assessment_geometry["street_segment_id"]) == proposed_native_ids
assert set(assessment_edges["street_segment_id"]) == proposed_native_ids
assert {"u", "v", "key"}.issubset(assessment_edges.columns)

files_to_create = [
    "native_segments.geojson",
    "native_input_support.parquet",
    "directed_edge_bridge.parquet",
    "proposed_history_support_check.parquet",
]
for filename in files_to_create:
    if (ASSESSMENT_DIR / filename).exists():
        raise FileExistsError(filename)

assessment_geometry.to_crs("EPSG:4326").to_file(
    ASSESSMENT_DIR / "native_segments.geojson",
    driver="GeoJSON",
)
native_support.to_parquet(
    ASSESSMENT_DIR / "native_input_support.parquet", index=False
)
assessment_edges.to_parquet(
    ASSESSMENT_DIR / "directed_edge_bridge.parquet", index=False
)
history_support_check.to_parquet(
    ASSESSMENT_DIR / "proposed_history_support_check.parquet", index=False
)

print("Native geometries saved:", len(assessment_geometry))
print("Native input-support rows saved:", len(native_support))
print("Directed-edge rows saved:", len(assessment_edges))
print("Proposed history boundary checks saved:", len(history_support_check))
print("Geometry CRS: EPSG:4326; coordinate order: longitude, latitude.")
print("History evidence describes the proposal, not a trained release.")

Native geometries saved: 460
Native input-support rows saved: 460
Directed-edge rows saved: 721
Proposed history boundary checks saved: 920
Geometry CRS: EPSG:4326; coordinate order: longitude, latitude.
History evidence describes the proposal, not a trained release.


In [32]:
assessment_settings = {
    "assessment_id": "pre2024-mobility-assessment-v1",
    "request_id": request_spec["request_id"],
    "status": "proposed_coverage_pending_review",
    "scenario": {
        "start_utc_inclusive": START_UTC.isoformat(),
        "end_utc_exclusive": END_UTC.isoformat(),
        "calendar_timezone": "America/New_York",
        "prediction_starts": len(requested_hours),
    },
    "mapping": {
        "metric_crs": METRIC_CRS,
        "search_distance_m": SEARCH_DISTANCE_M,
        "overlap_buffer_m": OVERLAP_BUFFER_M,
        "strong_mutual_overlap_percent": STRONG_OVERLAP_PERCENT,
        "split_evidence": (
            "Native coverage >=95%, combined mobility coverage >=95%, "
            "and connectivity through shared OSM nodes."
        ),
        "shared_native_relationships": "retained_and_flagged_for_review",
        "direction_equivalence_verified": False,
        "crosswalk_approved": False,
    },
    "proposed_time_policy": {
        "training_start_utc": TRAIN_START_UTC.isoformat(),
        "training_end_utc_exclusive": TRAIN_END_UTC.isoformat(),
        "training_label_available_by": DEV_START_UTC.isoformat(),
        "development_start_utc": DEV_START_UTC.isoformat(),
        "development_end_utc_exclusive": DEV_END_UTC.isoformat(),
        "development_label_available_by": MODEL_FREEZE_UTC.isoformat(),
        "label_rule": "target_end + 7 days <= relevant availability cutoff",
        "availability_verified_historically": False,
    },
    "proposed_history": {
        "lookback_days": 365,
        "delay_days": 7,
        "cutoff": "min(prediction_time - 7 days, training_end)",
        "event_pool": "eligible_training_positive_segment_hours_only",
        "event_count": len(proposed_history_events),
        "development_and_2024_events_included": False,
    },
    "rainfall_availability_lag_hours_assumed": 3,
    "missing_inputs": "unavailable_never_zero_filled",
    "new_model_fitted": False,
    "year_long_prediction_export_created": False,
}

with (ASSESSMENT_DIR / "assessment_settings.json").open("x") as file:
    json.dump(assessment_settings, file, indent=2)
    file.write("\n")

print("Assessment settings saved.")
print("Training dates remain proposed; no model experiment frozen.")
print("Next: source fingerprints, summary and README.")

Assessment settings saved.
Training dates remain proposed; no model experiment frozen.
Next: source fingerprints, summary and README.


In [33]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


dataset_manifest_path = (
    PROJECT_ROOT / "data/processed/baseline_dataset_v1/manifest.json"
)
dataset_manifest = json.loads(dataset_manifest_path.read_text())

source_paths = [
    dataset_manifest_path,
    climate_geometry_path,
    PROJECT_ROOT / "data/processed/baseline_street_features_v1/street_features.parquet",
    PROJECT_ROOT / "data/processed/physical_street_index_v1.parquet",
    PROJECT_ROOT / release_config["bundle_path"] / "street_cell_bridge.parquet",
    PROJECT_ROOT / "scripts/climate_history_features.py",
]

for partition in weather_partitions:
    source_paths.extend([
        partition["feature_path"], partition["manifest_path"]
    ])

source_fingerprints = {}

for path in source_paths:
    path = Path(path).resolve()
    actual_hash = sha256_file(path)

    relative_path = str(path.relative_to(PROJECT_ROOT))
    expected_hash = dataset_manifest["input_sha256"].get(
        str(path),
        dataset_manifest["input_sha256"].get(relative_path),
    )

    if expected_hash is not None and actual_hash != expected_hash:
        raise ValueError(f"Prepared source changed: {relative_path}")

    source_fingerprints[relative_path] = actual_hash

with (ASSESSMENT_DIR / "source_fingerprints.json").open("x") as file:
    json.dump(source_fingerprints, file, indent=2)
    file.write("\n")

print("Source fingerprints saved:", len(source_fingerprints))
print("Available frozen-dataset hash references matched.")
print("Raw rainfall source hashes remain in rainfall_source_ledger.csv.")
print("No source assets or model files changed.")

Source fingerprints saved: 32
Available frozen-dataset hash references matched.
Raw rainfall source hashes remain in rainfall_source_ledger.csv.
No source assets or model files changed.


In [34]:
coverage_summary = {
    "assessment_id": assessment_settings["assessment_id"],
    "status": "proposed_coverage_pending_review",
    "requested_physical_roads": len(road_assessment),
    "requested_mobility_contexts": int(
        road_assessment["mobility_context_count"].sum()
    ),
    "roads_with_draft_relationships": (
        draft_crosswalk["mobility_segment_id"].nunique()
    ),
    "roads_pending_mapping_review": int(
        road_assessment["proposed_native_segments"].eq(0).sum()
    ),
    "supplied_geometry_excluded_roads": len(geometry_exclusions),
    "contexts_on_geometry_excluded_roads": excluded_context_count,
    "draft_mapping_pairs": len(draft_crosswalk),
    "distinct_proposed_native_segments": len(native_support),
    "roads_requiring_shared_native_review": int(
        road_assessment["shared_native_review_required"].sum()
    ),
    "selected_radar_cells": len(selected_cells),
    "prediction_starts_per_segment": len(requested_hours),
    "valid_rainfall_cell_hours": int(
        scenario_rainfall["rainfall_inputs_valid"].sum()
    ),
    "unavailable_rainfall_cell_hours": len(unavailable_cell_hours),
    "hours_with_unavailable_inputs": len(unavailable_prediction_hours),
    "verified_raw_source_hours": int(
        rainfall_source_ledger["status"].eq("verified").sum()
    ),
    "declared_missing_raw_source_hours": len(required_missing_hours),
    "proposed_history_events": len(proposed_history_events),
    "new_model_fitted": False,
    "crosswalk_approved": False,
    "year_long_prediction_export_created": False,
}

with (ASSESSMENT_DIR / "coverage_summary.json").open("x") as file:
    json.dump(coverage_summary, file, indent=2)
    file.write("\n")

display(pd.Series(coverage_summary, name="value").to_frame())
print("Coverage summary saved.")

,value
assessment_id,pre2024-mobility-assessment-v1
status,proposed_coverage_pending_review
requested_physical_roads,700
requested_mobility_contexts,1025
roads_with_draft_relationships,449
roads_pending_mapping_review,251
supplied_geometry_excluded_roads,3
contexts_on_geometry_excluded_roads,5
draft_mapping_pairs,461
distinct_proposed_native_segments,460


Coverage summary saved.


In [35]:
import shutil

FINAL_SPEC_PATH = (
    Path.home() / ".codex/attachments"
    / "075d897c-95ac-45df-980f-1f633d09f894/Pasted text.txt"
)

if not FINAL_SPEC_PATH.is_file():
    raise FileNotFoundError(FINAL_SPEC_PATH)

# Recheck the original request before copying it.
for name, expected in request_manifest["files"].items():
    if sha256_file(REQUEST_DIR / name) != expected["sha256"]:
        raise ValueError(f"Request source changed: {name}")

request_reference_dir = ASSESSMENT_DIR / "request_reference"
request_reference_dir.mkdir(exist_ok=False)

for name in ["manifest.json", *request_manifest["files"]]:
    shutil.copy2(
        REQUEST_DIR / name,
        request_reference_dir / name,
    )

shutil.copy2(
    FINAL_SPEC_PATH,
    request_reference_dir / "final_model_specification.txt",
)

request_provenance = {
    "instruction_precedence": (
        "final_model_specification.txt supersedes the original "
        "package instruction to reuse the existing trained model"
    ),
    "files": {
        path.name: sha256_file(path)
        for path in sorted(request_reference_dir.iterdir())
        if path.is_file()
    },
}

with (ASSESSMENT_DIR / "request_provenance.json").open("x") as file:
    json.dump(request_provenance, file, indent=2)
    file.write("\n")

print("Original request and final model specification copied.")
print("Reference files:", len(request_provenance["files"]))
print("Instruction precedence recorded.")
print("Next: README and final package verification.")

Original request and final model specification copied.
Reference files: 8
Instruction precedence recorded.
Next: README and final package verification.


In [36]:
review_manifest_path = ASSESSMENT_DIR / "review_manifest.json"

delivered_files = {
    str(path.relative_to(ASSESSMENT_DIR)): sha256_file(path)
    for path in sorted(ASSESSMENT_DIR.rglob("*"))
    if path.is_file() and path != review_manifest_path
}

review_manifest = {
    "assessment_id": assessment_settings["assessment_id"],
    "status": "assessment_complete_mapping_approval_pending",
    "new_model_fitted": False,
    "files": delivered_files,
}

with review_manifest_path.open("x") as file:
    json.dump(review_manifest, file, indent=2)
    file.write("\n")

saved_manifest = json.loads(review_manifest_path.read_text())

for name, expected_hash in saved_manifest["files"].items():
    assert sha256_file(ASSESSMENT_DIR / name) == expected_hash, name

print("Delivered files verified:", len(delivered_files))
print("Assessment package:", ASSESSMENT_DIR)
print("Mapping approval and the new model fit remain pending.")

Delivered files verified: 27
Assessment package: /Users/macbookairm3/Documents/Plexa-A-Multi-Modal-Smart-City-Digital-Twin-and-Cascading-Resilience-Engine/data/processed/climate_coverage_assessments/pre2024-mobility-assessment-v1
Mapping approval and the new model fit remain pending.


## Approved Baseline — Notebook Training

Use the approved pre-2024 settings and verified training data.
Fit and save a separate notebook-controlled candidate.

The earlier script candidate remains preserved.

In [37]:
PRE24_CONFIG_PATH = (
    PROJECT_ROOT / "config/climate_pre2024_experiment_v1.json"
)

PRE24_SOURCE_RUN = (
    PROJECT_ROOT / "data/processed/climate_pre2024_runs/baseline-v1"
)

PRE24_NOTEBOOK_RUN = (
    PROJECT_ROOT / "data/processed/climate_pre2024_runs/notebook-v1"
)

pre24_config = json.loads(PRE24_CONFIG_PATH.read_text())

if PRE24_CONFIG_PATH.read_bytes() != (
    PRE24_SOURCE_RUN / "experiment.json"
).read_bytes():
    raise ValueError("Frozen configuration differs from the prepared run.")

print("Experiment:", pre24_config["experiment_id"])
print("Training interval:", pre24_config["training_interval"])
print("Development interval:", pre24_config["development_interval"])
print("Model features:", len(pre24_config["features"]))
print("Notebook run:", PRE24_NOTEBOOK_RUN)
print("Configuration loaded. No model loaded or fitted.")

Experiment: climate-pre2024-v1
Training interval: ['2020-10-14T04:00:00Z', '2023-06-24T04:00:00Z']
Development interval: ['2023-07-01T04:00:00Z', '2023-12-25T05:00:00Z']
Model features: 10
Notebook run: /Users/macbookairm3/Documents/Plexa-A-Multi-Modal-Smart-City-Digital-Twin-and-Cascading-Resilience-Engine/data/processed/climate_pre2024_runs/notebook-v1
Configuration loaded. No model loaded or fitted.


In [38]:
import hashlib

pre24_source_checkpoint = json.loads(
    (PRE24_SOURCE_RUN / "checkpoint_manifest.json").read_text()
)

pre24_sample_path = PRE24_SOURCE_RUN / "training_sample.parquet"
expected_hash = pre24_source_checkpoint["assets"][
    "training_sample.parquet"
]["sha256"]

with pre24_sample_path.open("rb") as file:
    actual_hash = hashlib.file_digest(file, "sha256").hexdigest()

if actual_hash != expected_hash:
    raise ValueError("Prepared training sample hash mismatch.")

pre24_training = pd.read_parquet(pre24_sample_path)

assert len(pre24_training) == pre24_source_checkpoint["training_rows"]
assert not pre24_training.duplicated(
    ["street_segment_id", "prediction_time_utc"]
).any()

display(
    pre24_training.groupby("label").agg(
        rows=("label", "size"),
        minimum_weight=("sample_weight", "min"),
        maximum_weight=("sample_weight", "max"),
    )
)

print("Training sample hash verified.")
print("Rows loaded:", len(pre24_training))
print("No saved model loaded and no fitting performed.")

,rows,minimum_weight,maximum_weight
label,,,
0,416986,5000.0,5000.0
1,7843,1.0,1.0


Training sample hash verified.
Rows loaded: 424829
No saved model loaded and no fitting performed.


In [39]:
import numpy as np

PRE24_FEATURES = pre24_config["features"]

X_pre24 = pre24_training[PRE24_FEATURES].astype("float64")
y_pre24 = pre24_training["label"].copy()
w_pre24 = pre24_training["sample_weight"].astype("float64")

assert np.isfinite(X_pre24.to_numpy()).all()
assert y_pre24.isin([0, 1]).all()
assert w_pre24.eq(np.where(y_pre24.eq(1), 1.0, 5000.0)).all()

train_start, train_end = map(
    pd.Timestamp, pre24_config["training_interval"]
)
available_by = pd.Timestamp(
    pre24_config["partition_label_cutoffs"]["training"]
)

prediction_times = pd.to_datetime(
    pre24_training["prediction_time_utc"], utc=True
)

assert prediction_times.ge(train_start).all()
assert prediction_times.lt(train_end).all()
assert (
    prediction_times + pd.Timedelta(hours=1, days=7)
).le(available_by).all()

assert (
    X_pre24["segment_positive_hours_365d"]
    <= X_pre24["cell_positive_hours_365d"]
).all()

print("Feature matrix:", X_pre24.shape)
print("Feature order:", PRE24_FEATURES)
print("Positive rows:", int(y_pre24.sum()))
print("Weights: raw inverse inclusion weights; no normalization.")
print("Training boundaries and assumed label availability verified.")
print("Input checks passed. No fitting performed.")

Feature matrix: (424829, 10)
Feature order: ['elevation_mean_m', 'terrain_endpoint_grade_abs', 'terrain_relief_m', 'length_m', 'rain_1h_mm', 'rain_3h_mm', 'rain_6h_mm', 'rain_24h_mm', 'segment_positive_hours_365d', 'cell_positive_hours_365d']
Positive rows: 7843
Weights: raw inverse inclusion weights; no normalization.
Training boundaries and assumed label availability verified.
Input checks passed. No fitting performed.


In [40]:
import sys
from importlib.metadata import version
from sklearn.ensemble import HistGradientBoostingClassifier

if sys.version.split()[0] != pre24_source_checkpoint["python_version"]:
    raise ValueError("Python version differs from the verified environment.")

for package, expected in pre24_source_checkpoint["package_versions"].items():
    if version(package) != expected:
        raise ValueError(f"Package version mismatch: {package}")

pre24_parameters = pre24_config["proposed_tree_parameters"].copy()

display(pd.Series(
    pre24_parameters, name="approved_value"
).to_frame())

pre24_model = HistGradientBoostingClassifier(**pre24_parameters)

print("Python and package versions match.")
print("New estimator created from frozen parameters.")
print("No previous model weights loaded and no fitting performed.")

,approved_value
class_weight,None
early_stopping,False
l2_regularization,1000.0
learning_rate,0.1
loss,log_loss
max_iter,100
max_leaf_nodes,15
min_samples_leaf,100
random_state,42


Python and package versions match.
New estimator created from frozen parameters.
No previous model weights loaded and no fitting performed.


## Model Fitting

Fit the approved histogram gradient boosting baseline using the verified
training matrix and raw inverse inclusion weights.

Development data is not used during fitting.

In [41]:
import time
import warnings
from threadpoolctl import threadpool_limits

if hasattr(pre24_model, "n_iter_"):
    raise RuntimeError("This estimator is already fitted; avoid accidental refitting.")

print("Fitting the pre-2024 baseline...")
started = time.perf_counter()

with warnings.catch_warnings(record=True) as pre24_fit_warnings:
    warnings.simplefilter("always")

    with threadpool_limits(limits=2):
        pre24_model.fit(
            X_pre24,
            y_pre24,
            sample_weight=w_pre24,
        )

pre24_fit_seconds = time.perf_counter() - started

assert list(pre24_model.classes_) == [0, 1]
assert list(pre24_model.feature_names_in_) == PRE24_FEATURES

print(f"Fit time: {pre24_fit_seconds:.2f} seconds")
print("Boosting iterations:", pre24_model.n_iter_)
print("Training rows:", len(X_pre24))
print("Positive rows:", int(y_pre24.sum()))

for warning in pre24_fit_warnings:
    print("Fit warning:", str(warning.message))

if not pre24_fit_warnings:
    print("No fit warnings captured.")

print("Model fitted in this notebook.")
print("Checkpoint saving and development evaluation remain pending.")

Fitting the pre-2024 baseline...
Fit time: 2.31 seconds
Boosting iterations: 100
Training rows: 424829
Positive rows: 7843
No fit warnings captured.
Model fitted in this notebook.
Checkpoint saving and development evaluation remain pending.


In [42]:
pre24_diagnostic_indices = np.linspace(
    0, len(X_pre24) - 1, min(10_000, len(X_pre24)),
    dtype=int,
)

with threadpool_limits(limits=2):
    pre24_diagnostic_scores = pre24_model.predict_proba(
        X_pre24.iloc[pre24_diagnostic_indices]
    )[:, 1]

assert np.isfinite(pre24_diagnostic_scores).all()
assert (
    (pre24_diagnostic_scores >= 0)
    & (pre24_diagnostic_scores <= 1)
).all()

display(pd.DataFrame([{
    "diagnostic_rows": len(pre24_diagnostic_scores),
    "minimum_score": pre24_diagnostic_scores.min(),
    "median_score": np.median(pre24_diagnostic_scores),
    "maximum_score": pre24_diagnostic_scores.max(),
    "distinct_scores": np.unique(pre24_diagnostic_scores).size,
    "exact_zero_scores": (pre24_diagnostic_scores == 0).sum(),
    "exact_one_scores": (pre24_diagnostic_scores == 1).sum(),
}]))

print("Training-input diagnostic only—not evaluation metrics.")
print("No threshold selected and no development data queried.")

,diagnostic_rows,minimum_score,median_score,maximum_score,distinct_scores,exact_zero_scores,exact_one_scores
0,10000,5.993473e-07,0.000001,0.000692,7070,0,0


Training-input diagnostic only—not evaluation metrics.
No threshold selected and no development data queried.


In [43]:
import shutil
import joblib

pre24_assets_to_copy = [
    "experiment.json",
    "training_sample.parquet",
    "history_events.parquet",
    "street_cell_bridge.parquet",
    "pretraining_report.json",
]

# Verify the recorded input assets before copying.
for name in pre24_assets_to_copy:
    if name in pre24_source_checkpoint["assets"]:
        actual = hashlib.sha256(
            (PRE24_SOURCE_RUN / name).read_bytes()
        ).hexdigest()

        if actual != pre24_source_checkpoint["assets"][name]["sha256"]:
            raise ValueError(f"Source asset changed: {name}")

# Preserve existing runs rather than overwriting them.
PRE24_NOTEBOOK_RUN.mkdir(parents=True, exist_ok=False)

for name in pre24_assets_to_copy:
    shutil.copy2(
        PRE24_SOURCE_RUN / name,
        PRE24_NOTEBOOK_RUN / name,
    )

joblib.dump(
    pre24_model,
    PRE24_NOTEBOOK_RUN / "history_tree.joblib",
)

print("Notebook model saved:", PRE24_NOTEBOOK_RUN)
print("Training sample, experiment and history assets preserved.")
print("Reload verification and checkpoint metadata are next.")

Notebook model saved: /Users/macbookairm3/Documents/Plexa-A-Multi-Modal-Smart-City-Digital-Twin-and-Cascading-Resilience-Engine/data/processed/climate_pre2024_runs/notebook-v1
Training sample, experiment and history assets preserved.
Reload verification and checkpoint metadata are next.


In [44]:
pre24_reloaded_model = joblib.load(
    PRE24_NOTEBOOK_RUN / "history_tree.joblib"
)

assert list(pre24_reloaded_model.classes_) == [0, 1]
assert list(pre24_reloaded_model.feature_names_in_) == PRE24_FEATURES

for name, expected in pre24_parameters.items():
    assert pre24_reloaded_model.get_params()[name] == expected, name

with threadpool_limits(limits=2):
    pre24_reloaded_scores = pre24_reloaded_model.predict_proba(
        X_pre24.iloc[pre24_diagnostic_indices]
    )[:, 1]

np.testing.assert_array_equal(
    pre24_diagnostic_scores,
    pre24_reloaded_scores,
)

print("Saved model classes, feature order and parameters verified.")
print("Maximum reload difference:",
      float(np.max(np.abs(
          pre24_diagnostic_scores - pre24_reloaded_scores
      ))))
print("Saved predictions match exactly.")
print("No retraining or development evaluation performed.")

Saved model classes, feature order and parameters verified.
Maximum reload difference: 0.0
Saved predictions match exactly.
No retraining or development evaluation performed.


In [45]:
from copy import deepcopy

pre24_checkpoint = {
    name: deepcopy(pre24_source_checkpoint[name])
    for name in [
        "features",
        "python_version",
        "package_versions",
        "history_helper",
        "preparation_evidence_sha256",
    ]
}

pre24_checkpoint.update({
    "run_id": "climate-pre2024-notebook-v1",
    "experiment_id": pre24_config["experiment_id"],
    "status": "notebook_fitted_development_pending",
    "execution_mode": "notebook_controlled_fit",
    "classes": pre24_reloaded_model.classes_.tolist(),
    "training_rows": len(X_pre24),
    "training_positive_rows": int(y_pre24.eq(1).sum()),
    "training_negative_rows": int(y_pre24.eq(0).sum()),
    "fit_seconds": pre24_fit_seconds,
    "iterations": int(pre24_reloaded_model.n_iter_),
    "fit_warnings": [str(w.message) for w in pre24_fit_warnings],
    "sampling_weights": "raw_inverse_inclusion_1_and_5000",
    "pretraining_report_project_path": str(
        (PRE24_NOTEBOOK_RUN / "pretraining_report.json")
        .relative_to(PROJECT_ROOT)
    ),
    "no_2024_outcomes_used": True,
    "fresh_independent_test_performed": False,
})

print("Notebook checkpoint metadata prepared.")
print("Run:", pre24_checkpoint["run_id"])
print("Training rows:", pre24_checkpoint["training_rows"])
print("File fingerprints and metadata saving are next.")

Notebook checkpoint metadata prepared.
Run: climate-pre2024-notebook-v1
Training rows: 424829
File fingerprints and metadata saving are next.


In [46]:
pre24_asset_names = [
    "history_tree.joblib",
    "experiment.json",
    "training_sample.parquet",
    "history_events.parquet",
    "street_cell_bridge.parquet",
]

pre24_checkpoint["assets"] = {
    name: {"sha256": sha256_file(PRE24_NOTEBOOK_RUN / name)}
    for name in pre24_asset_names
}

report_hash = sha256_file(
    PRE24_NOTEBOOK_RUN / "pretraining_report.json"
)
assert report_hash == pre24_checkpoint[
    "preparation_evidence_sha256"
]["pretraining_report"]

helper_path = (
    PROJECT_ROOT / pre24_checkpoint["history_helper"]["project_path"]
)
assert sha256_file(helper_path) == (
    pre24_checkpoint["history_helper"]["sha256"]
)

pre24_checkpoint_path = (
    PRE24_NOTEBOOK_RUN / "checkpoint_manifest.json"
)

with pre24_checkpoint_path.open("x") as file:
    json.dump(pre24_checkpoint, file, indent=2, allow_nan=False)
    file.write("\n")

print("Checkpoint manifest saved:", pre24_checkpoint_path)
print("Model and supporting asset fingerprints recorded.")
print("Preparation report and history helper hashes verified.")
print("Development evaluation has not started for this notebook run.")

Checkpoint manifest saved: /Users/macbookairm3/Documents/Plexa-A-Multi-Modal-Smart-City-Digital-Twin-and-Cascading-Resilience-Engine/data/processed/climate_pre2024_runs/notebook-v1/checkpoint_manifest.json
Model and supporting asset fingerprints recorded.
Preparation report and history helper hashes verified.
Development evaluation has not started for this notebook run.


## Development Evaluation — Initial Benchmark

Evaluate one window from the approved pre-2024 development period.
Check runtime and checkpointing before processing the full population.

In [47]:
import os
import subprocess

pre24_benchmark_command = [
    sys.executable,
    str(PROJECT_ROOT / "scripts/validate_climate_history.py"),
    "--run-dir", str(PRE24_NOTEBOOK_RUN),
    "--dataset-dir", str(
        PROJECT_ROOT / "data/processed/baseline_dataset_pre2024_v1"
    ),
    "--max-windows", "1",
]

started = time.perf_counter()

pre24_benchmark = subprocess.run(
    pre24_benchmark_command,
    cwd=PROJECT_ROOT,
    env=dict(os.environ, LOKY_MAX_CPU_COUNT="2"),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print(pre24_benchmark.stdout)
pre24_benchmark.check_returncode()

print(f"Total benchmark command time: {time.perf_counter() - started:.2f}s")
print("One development window only; no threshold selected.")

1/178 windows verified; 1,837,900 rows; latest window 3.46s
PARTIAL: /Users/macbookairm3/Documents/Plexa-A-Multi-Modal-Smart-City-Digital-Twin-and-Cascading-Resilience-Engine/data/processed/climate_pre2024_runs/notebook-v1/development_validation

Total benchmark command time: 6.38s
One development window only; no threshold selected.


In [48]:
pre24_evaluation_command = pre24_benchmark_command[:-2]

with subprocess.Popen(
    pre24_evaluation_command,
    cwd=PROJECT_ROOT,
    env=dict(os.environ, LOKY_MAX_CPU_COUNT="2"),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
) as process:
    for line in process.stdout:
        print(line, end="", flush=True)

    return_code = process.wait()

if return_code != 0:
    raise RuntimeError(
        f"Development evaluation stopped with exit code {return_code}."
    )

10/178 windows verified; 19,114,169 rows; latest window 0.73s
20/178 windows verified; 40,709,496 rows; latest window 3.64s
30/178 windows verified; 60,283,152 rows; latest window 3.72s
40/178 windows verified; 82,246,057 rows; latest window 3.77s
50/178 windows verified; 103,749,494 rows; latest window 3.51s
60/178 windows verified; 125,528,609 rows; latest window 3.72s
70/178 windows verified; 147,307,724 rows; latest window 3.73s
80/178 windows verified; 168,903,069 rows; latest window 3.18s
90/178 windows verified; 190,774,081 rows; latest window 3.72s
100/178 windows verified; 210,164,481 rows; latest window 3.47s
110/178 windows verified; 232,035,491 rows; latest window 3.59s
120/178 windows verified; 253,447,032 rows; latest window 3.72s
130/178 windows verified; 274,950,468 rows; latest window 3.73s
140/178 windows verified; 294,432,208 rows; latest window 3.74s
150/178 windows verified; 316,211,327 rows; latest window 3.53s
160/178 windows verified; 337,806,654 rows; latest wi

In [49]:
PRE24_RELEASE_DIR = PROJECT_ROOT / "models/climate_pre2024_v1"

pre24_package_command = [
    sys.executable,
    str(PROJECT_ROOT / "scripts/package_climate_pre2024.py"),
    "--run-dir", str(PRE24_NOTEBOOK_RUN),
    "--output-dir", str(PRE24_RELEASE_DIR),
]

pre24_packaging = subprocess.run(
    pre24_package_command,
    cwd=PROJECT_ROOT,
    env=dict(os.environ, LOKY_MAX_CPU_COUNT="2"),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print(pre24_packaging.stdout)
pre24_packaging.check_returncode()

{
  "package": "/Users/macbookairm3/Documents/Plexa-A-Multi-Modal-Smart-City-Digital-Twin-and-Cascading-Resilience-Engine/models/climate_pre2024_v1",
  "metrics": {
    "F1": 0.00819277108433735,
    "FN": 2053,
    "FP": 6179,
    "TN": 375107788,
    "TP": 34,
    "alerts": 6213,
    "approximate_AP": 0.00041038270815205483,
    "mean_alerts_per_24h": 35.093433749117445,
    "mean_false_alerts_per_24h": 34.901388562014596,
    "negative_rows": 375113967,
    "positive_rows": 2087,
    "precision": 0.005472396587799775,
    "recall": 0.01629132726401533,
    "rows": 375116054,
    "threshold": 0.0006100254940772366
  },
  "normalized_weather_rows": 2055456,
  "selected_cells": 234,
  "Ida_successful_rows": 192
}



In [50]:
pre24_verification = subprocess.run(
    [
        sys.executable,
        str(PRE24_RELEASE_DIR / "verify_release.py"),
        "--release", str(PRE24_RELEASE_DIR),
    ],
    cwd=PROJECT_ROOT,
    env=dict(
        os.environ,
        LOKY_MAX_CPU_COUNT="2",
        PYTHONDONTWRITEBYTECODE="1",
    ),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print(pre24_verification.stdout)
pre24_verification.check_returncode()

Handover and nested model/input hashes verified.
Approved scope verified: 441 roads / 666 contexts.
Standalone actual example replay: exact match.
441 available-hour predictions: OK; 441 March-gap records: explicit missing-feature errors.
Pending shared-native mobility ID rejected without fabricated output.
No fitting, model changes or 2024 outcome evaluation performed.



## Full-Scope Ida Replay — Approved 441 Roads

Prepare and verify Ida inputs for all approved native segments over
September 1–3, 2021, New York time, end exclusive.

Use the existing verified model and unchanged policies. Preserve the
current release and retain Ida’s training-overlap interpretation.

In [51]:
IDA_MODEL_DIR = (
    PROJECT_ROOT / "models/climate_pre2024_v1/model"
)

IDA_EXPECTED_MODEL_HASH = (
    "a28199bbefccbcf1683b4d4aec5268df2bc024f46d90f9a892870d2539aade44"
)

assert sha256_file(
    IDA_MODEL_DIR / "release_manifest.json"
) == IDA_EXPECTED_MODEL_HASH

IDA_START = pd.Timestamp("2021-09-01T04:00:00Z")
IDA_END = pd.Timestamp("2021-09-03T04:00:00Z")

ida_hours = pd.date_range(
    IDA_START, IDA_END, freq="h", inclusive="left"
)

IDA_BUNDLE_DIR = (
    PROJECT_ROOT
    / "data/processed/climate_scenarios/ida-pre2024-approved441-v1"
)

assert len(ida_hours) == 48

print("Existing model release hash verified.")
print("Ida prediction hours:", len(ida_hours))
print("Expected segment-hours:", 441 * len(ida_hours))
print("New bundle:", IDA_BUNDLE_DIR)
print("No files changed, predictions generated or fitting performed.")

Existing model release hash verified.
Ida prediction hours: 48
Expected segment-hours: 21168
New bundle: /Users/macbookairm3/Documents/Plexa-A-Multi-Modal-Smart-City-Digital-Twin-and-Cascading-Resilience-Engine/data/processed/climate_scenarios/ida-pre2024-approved441-v1
No files changed, predictions generated or fitting performed.


In [52]:
IDA_EXISTING_INPUT_DIR = IDA_MODEL_DIR.parent / "inputs_2024"

ida_input_manifest = json.loads(
    (IDA_EXISTING_INPUT_DIR / "input_manifest.json").read_text()
)

assert (
    ida_input_manifest["model_release_manifest_sha256"]
    == IDA_EXPECTED_MODEL_HASH
)

for name in [
    "approved_mobility_crosswalk.csv",
    "static_by_road.parquet",
    "supported_mobility_contexts.csv",
]:
    assert sha256_file(IDA_EXISTING_INPUT_DIR / name) == (
        ida_input_manifest["assets"][name]
    ), name

ida_crosswalk = pd.read_csv(
    IDA_EXISTING_INPUT_DIR / "approved_mobility_crosswalk.csv",
    dtype={"mobility_segment_id": str},
)
ida_static = pd.read_parquet(
    IDA_EXISTING_INPUT_DIR / "static_by_road.parquet"
)
ida_contexts = pd.read_csv(
    IDA_EXISTING_INPUT_DIR / "supported_mobility_contexts.csv",
    dtype=str,
)

assert len(ida_crosswalk) == 441
assert ida_crosswalk["street_segment_id"].nunique() == 441
assert ida_crosswalk["mobility_segment_id"].nunique() == 441
assert len(ida_contexts) == 666
assert set(ida_contexts["mobility_segment_id"]) == set(
    ida_crosswalk["mobility_segment_id"]
)
assert set(ida_static["street_segment_id"]) == set(
    ida_crosswalk["street_segment_id"]
)

print("Approved physical roads:", len(ida_crosswalk))
print("Mobility contexts:", len(ida_contexts))
print("Crosswalk, static inputs and context file hashes verified.")
print("Existing files unchanged; no predictions generated.")

Approved physical roads: 441
Mobility contexts: 666
Crosswalk, static inputs and context file hashes verified.
Existing files unchanged; no predictions generated.


In [53]:
ida_model_manifest = json.loads(
    (IDA_MODEL_DIR / "release_manifest.json").read_text()
)

assert sha256_file(
    IDA_MODEL_DIR / "street_cell_bridge.parquet"
) == ida_model_manifest["assets"]["street_cell_bridge.parquet"]

ida_model_bridge = pd.read_parquet(
    IDA_MODEL_DIR / "street_cell_bridge.parquet"
)

assert len(ida_static) == 441
assert not ida_static["street_segment_id"].duplicated().any()

ida_bridge_check = ida_static[
    ["street_segment_id", "grid_id", "cell_id"]
].merge(
    ida_model_bridge,
    on=["street_segment_id", "grid_id", "cell_id"],
    validate="one_to_one",
)

ida_terrain_features = [
    "elevation_mean_m", "terrain_endpoint_grade_abs",
    "terrain_relief_m", "length_m",
]

assert len(ida_bridge_check) == 441
assert ida_static["static_features_valid"].eq(True).all()
assert np.isfinite(
    ida_static[ida_terrain_features].to_numpy(dtype=float)
).all()
assert ida_static["length_m"].gt(0).all()

ida_cells = ida_bridge_check[["grid_id", "cell_id"]].drop_duplicates()

print("Native segments with verified static/model support:",
      len(ida_bridge_check))
print("Distinct radar cells:", len(ida_cells))
print("Static geometry remains a retrospective approximation.")
print("No model changed and no predictions generated.")

Native segments with verified static/model support: 441
Distinct radar cells: 234
Static geometry remains a retrospective approximation.
No model changed and no predictions generated.


In [54]:
IDA_WEATHER_DIR = (
    PROJECT_ROOT / "data/processed/baseline_weather_v1/2021-09"
)
IDA_WEATHER_PATH = IDA_WEATHER_DIR / "weather_features.parquet"

ida_provenance_path = IDA_MODEL_DIR / "source_provenance.json"
assert sha256_file(ida_provenance_path) == (
    ida_model_manifest["assets"]["source_provenance.json"]
)
ida_provenance = json.loads(ida_provenance_path.read_text())

ida_dataset_manifest_path = (
    PROJECT_ROOT / "data/processed/baseline_dataset_pre2024_v1/manifest.json"
)
assert sha256_file(ida_dataset_manifest_path) == (
    ida_provenance["prepared_dataset_manifest_sha256"]
)

ida_frozen_sources = json.loads(
    ida_dataset_manifest_path.read_text()
)["input_sha256"]

for path in [
    IDA_WEATHER_PATH,
    IDA_WEATHER_DIR / "manifest.json",
]:
    expected = ida_frozen_sources.get(
        str(path),
        ida_frozen_sources.get(str(path.relative_to(PROJECT_ROOT))),
    )
    if expected is None or sha256_file(path) != expected:
        raise ValueError(f"Ida rainfall source verification failed: {path.name}")

print("Ida rainfall partition and manifest hashes verified.")
print("Prepared sources match the model's recorded dataset.")
print("Next: load the 234 selected cells over all 48 hours.")

Ida rainfall partition and manifest hashes verified.
Prepared sources match the model's recorded dataset.
Next: load the 234 selected cells over all 48 hours.


In [55]:
assert ida_cells["grid_id"].nunique() == 1

ida_rain_features = [
    "rain_1h_mm", "rain_3h_mm", "rain_6h_mm", "rain_24h_mm"
]
ida_weather_keys = ["grid_id", "cell_id", "prediction_time_utc"]

ida_loaded_weather = pd.read_parquet(
    IDA_WEATHER_PATH,
    columns=ida_weather_keys + ida_rain_features + ["weather_features_valid"],
    filters=[
        ("grid_id", "==", ida_cells["grid_id"].iloc[0]),
        ("cell_id", "in", ida_cells["cell_id"].tolist()),
        ("prediction_time_utc", ">=", IDA_START.to_pydatetime()),
        ("prediction_time_utc", "<", IDA_END.to_pydatetime()),
    ],
)

assert not ida_loaded_weather.duplicated(ida_weather_keys).any()

ida_expected_cell_hours = ida_cells.merge(
    pd.DataFrame({"prediction_time_utc": ida_hours}),
    how="cross",
)

ida_weather = ida_expected_cell_hours.merge(
    ida_loaded_weather,
    on=ida_weather_keys,
    how="left",
    validate="one_to_one",
    indicator="source_row_status",
)

ida_rain_values = ida_weather[ida_rain_features].to_numpy(dtype=float)

ida_weather["rainfall_inputs_valid"] = (
    ida_weather["weather_features_valid"].eq(True)
    & np.isfinite(ida_rain_values).all(axis=1)
    & (ida_rain_values >= 0).all(axis=1)
    & (np.diff(ida_rain_values, axis=1) >= -1e-9).all(axis=1)
)

print("Expected cell-hours:", len(ida_expected_cell_hours))
print("Source rows present:", len(ida_loaded_weather))
print("Valid cell-hours:", int(ida_weather["rainfall_inputs_valid"].sum()))
print("Unavailable cell-hours:", int((~ida_weather["rainfall_inputs_valid"]).sum()))
print("No missing rainfall values replaced with zero.")

Expected cell-hours: 11232
Source rows present: 11232
Valid cell-hours: 11232
Unavailable cell-hours: 0
No missing rainfall values replaced with zero.


In [56]:
ida_segment_hours = ida_bridge_check.merge(
    pd.DataFrame({"prediction_time_utc": ida_hours}),
    how="cross",
)

ida_segment_coverage = ida_segment_hours.merge(
    ida_weather[
        ida_weather_keys + ["rainfall_inputs_valid", "source_row_status"]
    ],
    on=ida_weather_keys,
    how="left",
    validate="many_to_one",
)

assert len(ida_segment_coverage) == 21_168
assert not ida_segment_coverage.duplicated(
    ["street_segment_id", "prediction_time_utc"]
).any()

ida_road_coverage = ida_segment_coverage.groupby(
    "street_segment_id", as_index=False
).agg(
    expected_hours=("prediction_time_utc", "size"),
    valid_input_hours=("rainfall_inputs_valid", "sum"),
)

ida_road_coverage["unavailable_hours"] = (
    ida_road_coverage["expected_hours"]
    - ida_road_coverage["valid_input_hours"]
)

assert ida_road_coverage["expected_hours"].eq(48).all()

print("Physical segments checked:", len(ida_road_coverage))
print("Expected segment-hours:", len(ida_segment_coverage))
print("Valid segment-hours:",
      int(ida_road_coverage["valid_input_hours"].sum()))
print("Unavailable segment-hours:",
      int(ida_road_coverage["unavailable_hours"].sum()))
print("No predictions generated and no model changed.")


Physical segments checked: 441
Expected segment-hours: 21168
Valid segment-hours: 21168
Unavailable segment-hours: 0
No predictions generated and no model changed.


In [57]:
ida_weather_manifest = json.loads(
    (IDA_WEATHER_DIR / "manifest.json").read_text()
)

assert ida_weather_manifest["preparation_config"][
    "rainfall_availability_lag_hours"
] == 3

ida_source_ends = pd.date_range(
    IDA_START - pd.Timedelta(hours=26),
    ida_hours[-1] - pd.Timedelta(hours=3),
    freq="h",
)

ida_source_rows = []

for source_end in ida_source_ends:
    filename = f"radar_{source_end:%Y%m%d_%H%M%S}.parquet"
    path = PROJECT_ROOT / "data/raw/radar_parquet_v2" / filename
    expected_hash = ida_weather_manifest["source_sha256"].get(filename)

    if not path.is_file() or expected_hash is None:
        raise ValueError(f"Required source needs investigation: {filename}")

    actual_hash = sha256_file(path)
    if actual_hash != expected_hash:
        raise ValueError(f"Source hash mismatch: {filename}")

    ida_source_rows.append({
        "source_file": filename,
        "interval_start_utc": source_end - pd.Timedelta(hours=1),
        "interval_end_utc": source_end,
        "status": "verified",
        "sha256": actual_hash,
    })

ida_source_ledger = pd.DataFrame(ida_source_rows)

assert len(ida_source_ledger) == 71

print("Verified raw source hours:", len(ida_source_ledger))
print("First interval start:",
      ida_source_ledger["interval_start_utc"].min())
print("Last interval end:",
      ida_source_ledger["interval_end_utc"].max())
print("Actual publication times remain unverified; three-hour lag retained.")

Verified raw source hours: 71
First interval start: 2021-08-31 01:00:00+00:00
Last interval end: 2021-09-03 00:00:00+00:00
Actual publication times remain unverified; three-hour lag retained.


In [58]:
import shutil

ida_files_to_reuse = [
    "approved_mobility_crosswalk.csv",
    "static_by_road.parquet",
    "supported_mobility_contexts.csv",
    "native_segments.geojson",
    "directed_edge_bridge.parquet",
]

for name in ida_files_to_reuse:
    assert sha256_file(IDA_EXISTING_INPUT_DIR / name) == (
        ida_input_manifest["assets"][name]
    ), name

# Refuse to overwrite an existing bundle.
IDA_BUNDLE_DIR.mkdir(parents=True, exist_ok=False)
IDA_INPUT_DIR = IDA_BUNDLE_DIR / "inputs"
IDA_INPUT_DIR.mkdir()

for name in ida_files_to_reuse:
    shutil.copyfile(
        IDA_EXISTING_INPUT_DIR / name,
        IDA_INPUT_DIR / name,
    )

ida_weather.to_parquet(
    IDA_INPUT_DIR / "rainfall_by_cell_hour.parquet",
    index=False,
)
ida_source_ledger.to_csv(
    IDA_INPUT_DIR / "rainfall_source_ledger.csv",
    index=False,
)

print("Ida bundle created:", IDA_BUNDLE_DIR)
print("Static inputs and approved identities copied without modification.")
print("Rainfall cell-hours saved:", len(ida_weather))
print("Verified raw-source intervals saved:", len(ida_source_ledger))
print("No model files copied, changed or fitted.")

Ida bundle created: /Users/macbookairm3/Documents/Plexa-A-Multi-Modal-Smart-City-Digital-Twin-and-Cascading-Resilience-Engine/data/processed/climate_scenarios/ida-pre2024-approved441-v1
Static inputs and approved identities copied without modification.
Rainfall cell-hours saved: 11232
Verified raw-source intervals saved: 71
No model files copied, changed or fitted.


In [59]:
ida_hourly_coverage = ida_segment_coverage.groupby(
    "prediction_time_utc", as_index=False
).agg(
    expected_segments=("street_segment_id", "size"),
    valid_input_segments=("rainfall_inputs_valid", "sum"),
)

ida_hourly_coverage["unavailable_segments"] = (
    ida_hourly_coverage["expected_segments"]
    - ida_hourly_coverage["valid_input_segments"]
)

ida_unavailable_segments = ida_segment_coverage.loc[
    ~ida_segment_coverage["rainfall_inputs_valid"].eq(True)
].copy()

ida_segment_coverage.to_parquet(
    IDA_INPUT_DIR / "segment_hour_coverage.parquet", index=False
)
ida_road_coverage.to_csv(
    IDA_INPUT_DIR / "road_coverage.csv", index=False
)
ida_hourly_coverage.to_csv(
    IDA_INPUT_DIR / "hourly_coverage.csv", index=False
)
ida_unavailable_segments.to_parquet(
    IDA_INPUT_DIR / "unavailable_segment_hours.parquet", index=False
)
ida_hourly_coverage.loc[
    ida_hourly_coverage["unavailable_segments"].gt(0)
].to_csv(
    IDA_INPUT_DIR / "unavailable_prediction_hours.csv", index=False
)

assert len(ida_hourly_coverage) == 48
assert ida_hourly_coverage["expected_segments"].eq(441).all()

print("Coverage saved: 441 roads × 48 hours.")
print("Unavailable segment-hours:", len(ida_unavailable_segments))
print("No unavailable inputs omitted or zero-filled.")

Coverage saved: 441 roads × 48 hours.
Unavailable segment-hours: 0
No unavailable inputs omitted or zero-filled.


In [60]:
ida_bundle_manifest = {
    "input_id": "ida-pre2024-approved441-v1",
    "package_id": ida_model_manifest["package_id"],
    "model_release_manifest_sha256": IDA_EXPECTED_MODEL_HASH,
    "utc_interval": [
        IDA_START.isoformat().replace("+00:00", "Z"),
        IDA_END.isoformat().replace("+00:00", "Z"),
    ],
    "calendar_timezone": "America/New_York",
    "physical_roads": 441,
    "mobility_contexts": 666,
    "rainfall_cells": len(ida_cells),
    "hourly_starts": len(ida_hours),
    "expected_segment_hours": 21_168,
    "unavailable_segment_hours": len(ida_unavailable_segments),
    "rainfall_lag_hours_assumed": 3,
    "history_policy": ida_model_manifest["history_policy"],
    "ida_training_overlap": True,
    "contains_target_labels": False,
    "source_fingerprints": {
        "prepared_dataset_manifest": sha256_file(ida_dataset_manifest_path),
        "prepared_rainfall": sha256_file(IDA_WEATHER_PATH),
        "rainfall_manifest": sha256_file(IDA_WEATHER_DIR / "manifest.json"),
        "approved_source_input_manifest": sha256_file(
            IDA_EXISTING_INPUT_DIR / "input_manifest.json"
        ),
    },
    "assets": {
        path.name: sha256_file(path)
        for path in sorted(IDA_INPUT_DIR.iterdir())
        if path.is_file() and not path.name.startswith(".")
        and path.name != "input_manifest.json"
    },
}

with (IDA_INPUT_DIR / "input_manifest.json").open("x") as file:
    json.dump(ida_bundle_manifest, file, indent=2, allow_nan=False)
    file.write("\n")

print("Model-bound Ida input manifest saved.")
print("Existing model release unchanged.")
print("Next: actual full-scenario inference.")

Model-bound Ida input manifest saved.
Existing model release unchanged.
Next: actual full-scenario inference.


In [61]:
import sys

ida_runtime_files = {
    "predict_climate_history.py": "predict.py",
    "prepare_climate_pre2024_request.py": "prepare_request.py",
    "climate_history_features.py": "climate_history_features.py",
}

for project_file, bundle_file in ida_runtime_files.items():
    assert sha256_file(
        PROJECT_ROOT / "scripts" / project_file
    ) == ida_model_manifest["assets"][bundle_file], project_file

scripts_path = str(PROJECT_ROOT / "scripts")
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)

from predict_climate_history import ClimateHistoryPredictor
from prepare_climate_pre2024_request import prepare as ida_prepare_request

ida_predictor = ClimateHistoryPredictor(IDA_MODEL_DIR)

assert ida_predictor.manifest["package_id"] == (
    ida_bundle_manifest["package_id"]
)

print("Inference and history code match the verified release.")
print("Saved model, dependencies and bundled asset hashes verified.")
print("No fitting performed.")
print("Ready for full-scenario inference.")

Inference and history code match the verified release.
Saved model, dependencies and bundled asset hashes verified.
No fitting performed.
Ready for full-scenario inference.


In [62]:
ida_actual_records = []
ida_feature_lineage = []
ida_window_summary = []

for number, timestamp in enumerate(ida_hours, start=1):
    stamp = timestamp.isoformat().replace("+00:00", "Z")
    request, context = ida_prepare_request(
        IDA_INPUT_DIR, IDA_MODEL_DIR, stamp
    )

    request_bytes = (
        json.dumps(request, indent=2, allow_nan=False) + "\n"
    ).encode("utf-8")
    request_hash = hashlib.sha256(request_bytes).hexdigest()

    response = ida_predictor.predict(request, request_hash)

    assert len(request["records"]) == 441
    assert len(response["records"]) == 441
    assert {r["request_id"] for r in response["records"]} == {
        r["request_id"] for r in request["records"]
    }

    ida_actual_records.extend([
        {"prediction_time_utc": stamp, "request_sha256": request_hash, **r}
        for r in response["records"]
    ])
    ida_feature_lineage.extend([
        {"prediction_time_utc": stamp, **r}
        for r in context["records"]
    ])

    successful = [r for r in response["records"] if r["status"] == "ok"]
    ida_window_summary.append({
        "prediction_time_utc": stamp,
        "request_sha256": request_hash,
        "successful_records": len(successful),
        "error_records": 441 - len(successful),
        "actual_alerts": sum(r["result"]["model_alert"] for r in successful),
    })

    if number % 8 == 0:
        print(f"{number}/48 hours processed")

ida_inference_summary = pd.DataFrame(ida_window_summary)

assert len(ida_actual_records) == 21_168
display(ida_inference_summary[[
    "successful_records", "error_records", "actual_alerts"
]].sum().to_frame("total"))

print("Actual model outputs retained in memory; no threshold changed.")

8/48 hours processed
16/48 hours processed
24/48 hours processed
32/48 hours processed
40/48 hours processed
48/48 hours processed


,total
successful_records,21168
error_records,0
actual_alerts,0


Actual model outputs retained in memory; no threshold changed.


In [63]:
import pyarrow as pa
import pyarrow.parquet as pq

ida_result_files = [
    "actual_predictions.parquet",
    "feature_row_lineage.parquet",
    "inference_by_hour.csv",
]

for name in ida_result_files:
    if (IDA_BUNDLE_DIR / name).exists():
        raise FileExistsError(f"Result already exists: {name}")

pq.write_table(
    pa.Table.from_pylist(ida_actual_records),
    IDA_BUNDLE_DIR / "actual_predictions.parquet",
    compression="snappy",
)

pq.write_table(
    pa.Table.from_pylist(ida_feature_lineage),
    IDA_BUNDLE_DIR / "feature_row_lineage.parquet",
    compression="snappy",
)

ida_inference_summary.to_csv(
    IDA_BUNDLE_DIR / "inference_by_hour.csv",
    index=False,
)

assert pq.read_metadata(
    IDA_BUNDLE_DIR / "actual_predictions.parquet"
).num_rows == 21_168

print("Actual prediction records saved: 21,168")
print("Feature-lineage records saved:", len(ida_feature_lineage))
print("Hourly summaries saved:", len(ida_inference_summary))
print("Next: full-scenario replay verification.")

Actual prediction records saved: 21,168
Feature-lineage records saved: 21168
Hourly summaries saved: 48
Next: full-scenario replay verification.


In [64]:
ida_saved_records = pq.read_table(
    IDA_BUNDLE_DIR / "actual_predictions.parquet"
).to_pylist()

ida_saved_by_hour = {}
for record in ida_saved_records:
    ida_saved_by_hour.setdefault(
        record["prediction_time_utc"], []
    ).append(record)

ida_verified_records = 0
ida_training_end = pd.Timestamp(
    ida_predictor.experiment["training_interval"][1]
)

for number, timestamp in enumerate(ida_hours, start=1):
    stamp = timestamp.isoformat().replace("+00:00", "Z")
    request, _ = ida_prepare_request(
        IDA_INPUT_DIR, IDA_MODEL_DIR, stamp
    )
    raw = (json.dumps(request, indent=2, allow_nan=False) + "\n").encode()
    request_hash = hashlib.sha256(raw).hexdigest()

    response = ida_predictor.predict(request, request_hash)
    replay_records = [
        {"prediction_time_utc": stamp, "request_sha256": request_hash, **r}
        for r in response["records"]
    ]

    assert replay_records == ida_saved_by_hour[stamp], stamp

    expected_cutoff = min(
        timestamp - pd.Timedelta(days=7), ida_training_end
    )
    for record in response["records"]:
        if record["status"] == "ok":
            assert pd.Timestamp(
                record["result"]["history_cutoff_utc"]
            ) == expected_cutoff

    ida_verified_records += len(replay_records)
    if number % 8 == 0:
        print(f"{number}/48 hours replay-verified")

assert ida_verified_records == 21_168
assert len({r["request_id"] for r in ida_saved_records}) == 21_168

print("All 21,168 saved records match replay exactly.")
print("History cutoffs verified; no future history substituted.")

8/48 hours replay-verified
16/48 hours replay-verified
24/48 hours replay-verified
32/48 hours replay-verified
40/48 hours replay-verified
48/48 hours replay-verified
All 21,168 saved records match replay exactly.
History cutoffs verified; no future history substituted.


In [65]:
import tempfile

ida_cli_checks = []

with tempfile.TemporaryDirectory(prefix="plexa-ida-check-") as temporary:
    temporary = Path(temporary)

    for boundary, timestamp in [
        ("first", ida_hours[0]),
        ("last", ida_hours[-1]),
    ]:
        stamp = timestamp.isoformat().replace("+00:00", "Z")
        request_path = temporary / f"{boundary}_request.json"
        result_path = temporary / f"{boundary}_result.json"

        completed = subprocess.run(
            [
                sys.executable, str(IDA_MODEL_DIR / "prepare_request.py"),
                "--bundle", str(IDA_MODEL_DIR),
                "--inputs", str(IDA_INPUT_DIR),
                "--time", stamp,
                "--request", str(request_path),
                "--result", str(result_path),
            ],
            cwd=temporary,
            env=dict(os.environ, LOKY_MAX_CPU_COUNT="2",
                     PYTHONDONTWRITEBYTECODE="1"),
            capture_output=True, text=True,
        )

        if completed.returncode != 0:
            raise RuntimeError(completed.stdout + completed.stderr)

        actual = json.loads(result_path.read_text())
        expected = [
            {k: v for k, v in r.items()
             if k not in ["prediction_time_utc", "request_sha256"]}
            for r in ida_saved_by_hour[stamp]
        ]

        assert actual["records"] == expected
        assert actual["request_sha256"] == (
            ida_saved_by_hour[stamp][0]["request_sha256"]
        )

        ida_cli_checks.append({
            "boundary": boundary,
            "prediction_time_utc": stamp,
            "records": len(actual["records"]),
            "exact_match": True,
        })

display(pd.DataFrame(ida_cli_checks))
print("Standalone boundary preparation and inference verified.")

,boundary,prediction_time_utc,records,exact_match
0,first,2021-09-01T04:00:00Z,441,True
1,last,2021-09-03T03:00:00Z,441,True


Standalone boundary preparation and inference verified.


In [66]:
ida_successful_records = [
    r for r in ida_saved_records if r["status"] == "ok"
]
ida_actual_alert_count = sum(
    r["result"]["model_alert"] for r in ida_successful_records
)

assert sha256_file(
    IDA_MODEL_DIR / "release_manifest.json"
) == IDA_EXPECTED_MODEL_HASH

assert all(
    r["result"]["model_alert_threshold"]
    == ida_model_manifest["alert_threshold"]
    for r in ida_successful_records
)

ida_verification_report = {
    "bundle_id": "ida-pre2024-approved441-v1",
    "model_release_manifest_sha256": IDA_EXPECTED_MODEL_HASH,
    "expected_segment_hours": 21_168,
    "successful_records": len(ida_successful_records),
    "error_records": len(ida_saved_records) - len(ida_successful_records),
    "actual_alerts": int(ida_actual_alert_count),
    "unavailable_input_segment_hours": len(ida_unavailable_segments),
    "verified_raw_source_hours": len(ida_source_ledger),
    "full_scenario_replay_exact": True,
    "history_cutoffs_verified": True,
    "standalone_boundary_checks": ida_cli_checks,
    "model_threshold": ida_model_manifest["alert_threshold"],
    "model_changed": False,
    "new_fit_performed": False,
    "ida_training_overlap": True,
    "independent_evaluation": False,
}

with (IDA_BUNDLE_DIR / "verification_report.json").open("x") as file:
    json.dump(ida_verification_report, file, indent=2, allow_nan=False)
    file.write("\n")

print("Verification report saved.")
print("Remaining: README, final manifest and notebook/Git review.")

Verification report saved.
Remaining: README, final manifest and notebook/Git review.


In [69]:
ida_readme_lines = [
    "# Full-Scope Ida Replay — Approved 441 Roads",
    "",
    "## Scope and results",
    "UTC window: 2021-09-01T04:00:00Z to 2021-09-03T04:00:00Z, end exclusive.",
    "441 physical segments, 666 mobility contexts, 48 hourly starts.",
    "21,168 successful predictions; zero errors; zero actual alerts.",
    "No unavailable input segment-hours. All 71 raw source hours verified.",
    "Full-scenario replay and standalone boundary checks passed.",
    "",
    "## Model binding",
    f"Package: {ida_model_manifest['package_id']}",
    f"Release-manifest SHA256: {IDA_EXPECTED_MODEL_HASH}",
    f"Unchanged threshold: {ida_model_manifest['alert_threshold']}",
    "Use the existing model release; no new fit or policy changes.",
    "",
    "## Files",
    "- inputs/: normalized inputs, crosswalk, geometry, source and coverage ledgers.",
    "- actual_predictions.parquet: all 21,168 actual native results.",
    "- feature_row_lineage.parquet: request identities and input provenance.",
    "- inference_by_hour.csv: request hashes and hourly result counts.",
    "- verification_report.json: completed verification checks.",
    "- bundle_manifest.json: delivered-file hashes.",
    "Prediction records retain nested results and warnings; top-level time and",
    "request_sha256 are additional storage/provenance fields.",
    "",
    "## Reproduce one hour",
    "Use the existing pinned model environment. From this bundle's root,",
    "replace MODEL_PATH with the existing model directory:",
    "",
    "    python MODEL_PATH/prepare_request.py --bundle MODEL_PATH --inputs inputs --time 2021-09-01T04:00:00Z --request replay_request.json --result replay_result.json",
    "",
    "Use fresh output filenames. Repeat for each hour in inference_by_hour.csv.",
    "",
    "## Interpretation",
    "Retain the three-hour rainfall availability assumption and 1/3/6/24-hour windows.",
    "History uses min(prediction_time - 7 days, training_end), then looks back 365 days.",
    "Ida uses its own past-only cutoffs, not the fixed context of the 2024 replay.",
    "Historic publication times remain unverified. Missing inputs are never zero-filled.",
    "Ida overlaps training: this is an integration replay, not independent evaluation.",
    "Scores are uncalibrated report proxies, not depth, closure or safe-passage predictions.",
    "Mobility directions are metadata, not independent climate predictions.",
    "The existing model and earlier releases remain unchanged.",
]

with (IDA_BUNDLE_DIR / "README.md").open("x") as file:
    file.write("\n".join(ida_readme_lines) + "\n")

print("README saved.")

README saved.


In [70]:
ida_bundle_manifest_path = IDA_BUNDLE_DIR / "bundle_manifest.json"

ida_delivered_files = {
    str(path.relative_to(IDA_BUNDLE_DIR)): sha256_file(path)
    for path in sorted(IDA_BUNDLE_DIR.rglob("*"))
    if path.is_file()
    and path != ida_bundle_manifest_path
    and not any(
        part.startswith(".")
        for part in path.relative_to(IDA_BUNDLE_DIR).parts
    )
}

ida_final_manifest = {
    "bundle_id": "ida-pre2024-approved441-v1",
    "model_package_id": ida_model_manifest["package_id"],
    "model_release_manifest_sha256": IDA_EXPECTED_MODEL_HASH,
    "expected_segment_hours": 21_168,
    "successful_records": ida_verification_report["successful_records"],
    "error_records": ida_verification_report["error_records"],
    "actual_alerts": ida_verification_report["actual_alerts"],
    "ida_training_overlap": True,
    "files": ida_delivered_files,
}

with ida_bundle_manifest_path.open("x") as file:
    json.dump(ida_final_manifest, file, indent=2, allow_nan=False)
    file.write("\n")

for name, expected_hash in ida_final_manifest["files"].items():
    assert sha256_file(IDA_BUNDLE_DIR / name) == expected_hash, name

assert sha256_file(
    IDA_MODEL_DIR / "release_manifest.json"
) == IDA_EXPECTED_MODEL_HASH

print("Delivered files verified:", len(ida_delivered_files))
print("Existing model binding unchanged.")
print("Complete Ida bundle:", IDA_BUNDLE_DIR)
print("21,168 successful records; zero errors; zero alerts.")

Delivered files verified: 18
Existing model binding unchanged.
Complete Ida bundle: /Users/macbookairm3/Documents/Plexa-A-Multi-Modal-Smart-City-Digital-Twin-and-Cascading-Resilience-Engine/data/processed/climate_scenarios/ida-pre2024-approved441-v1
21,168 successful records; zero errors; zero alerts.
